# Tier 2 — Electricity emission signals

Execute notebook 01 preprocessing first. The code below retains AEF, the four original UpRamp configurations and the original MSDR/DLR calculations. The published input selection is UpRamp `k3_q20` and MSDR Soft MEF `K=2`, `ROLL_HOURS=336`. The final cell maps the selected signal columns to the input names used by notebook 03. The full MSDR calculation fits rolling windows for both actual and forecast streams and can take substantial time.


In [ ]:
from pathlib import Path
import os

# Open from the repository root or its notebooks/ subfolder.
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "Csv files of input and output").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "Csv files of input and output").is_dir():
    raise FileNotFoundError("Set REPO_ROOT to the downloaded repository folder")
DATA = REPO_ROOT / "Csv files of input and output"
WORK = REPO_ROOT / "generated/upstream"
WORK.mkdir(parents=True, exist_ok=True)

USE_NEW_TIER1 = True  # True after notebook 01; False selects the archived forecasts.
TIER1_WORK = WORK / "tier1"
TIER2_WORK = WORK / "tier2"
TIER2_WORK.mkdir(parents=True, exist_ok=True)
EMBER_SOURCE = DATA / "emission tier2"
TECH_ARCHIVES = {
    "gen_fossil_coal_oil_mw": "coaloil", "gen_biomass_mw": "biomass",
    "gen_wind_onshore_mw": "wind", "gen_fossil_gas_mw": "fossil gas",
    "gen_nuclear_mw": "nuclear", "gen_solar_mw": "solar", "gen_waste_mw": "waste",
    "gen_other_renewable_mw": "other renawables", "gen_hydro_run_of_river_mw": "hydro run of river",
    "gen_other_mw": "other", "gen_hydro_dispatchable_mw": "hydro dispatchable",
}

def prediction_paths(tech):
    folder = TIER1_WORK / tech if USE_NEW_TIER1 else DATA / "results/ML/tier1" / TECH_ARCHIVES[tech]
    return sorted(str(p) for p in folder.glob("*hourly_predictions_2024*.csv"))


# 2. Tier 2 — dynamic day ahead electricity emission signals

This section keeps the final emission-signal construction blocks used in the paper:

- **AEF**
- **UpRamp MEF**
- **MSDR Soft MEF**


## 2.1 AEF from actual generation


In [ ]:
# ============================================
# TIER 2.1.A – Build hourly AEF from actual gen (2022–23 train + 2024 test)
# ============================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PROC_DIR = str(TIER1_WORK)
EMBER_CO2_DIR = str(EMBER_SOURCE)
TIER2_DIR     = str(TIER2_WORK)
TIER2_AEF_DIR = os.path.join(TIER2_DIR, "tier2_1_aef")

os.makedirs(TIER2_DIR, exist_ok=True)
os.makedirs(TIER2_AEF_DIR, exist_ok=True)

TRAIN_MASTER_PATH = os.path.join(DATA_PROC_DIR, "master_ES_train_2022_2023_interp.csv")
TEST_MASTER_PATH  = os.path.join(DATA_PROC_DIR, "master_ES_test_2024_interp.csv")

print("[AEF-1A] Loading master train/test...")
train_master = pd.read_csv(TRAIN_MASTER_PATH, parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()
test_master  = pd.read_csv(TEST_MASTER_PATH,  parse_dates=["datetime_utc"]).set_index("datetime_utc").sort_index()

print("[AEF-1A] TRAIN master:", train_master.shape,
      "| range:", train_master.index.min(), "->", train_master.index.max())
print("[AEF-1A] TEST  master:", test_master.shape,
      "| range:", test_master.index.min(), "->", test_master.index.max())


# ---------- EF map loader (kgCO2/kWh per Tier1 tech) ----------

def load_ef_map(ember_dir: str) -> dict:
    csvs = [
        os.path.join(ember_dir, f)
        for f in os.listdir(ember_dir)
        if f.startswith("gv2-tier2-ember-per-tech-emission-") and f.endswith(".csv")
    ]
    if not csvs:
        raise RuntimeError("[AEF-1A] No Ember EF files found")

    frames = []
    for path in sorted(csvs):
        df = pd.read_csv(path)
        if "Tier1 Tech" not in df.columns or "EF (kgCo2/MWh)" not in df.columns:
            raise ValueError(f"[AEF-1A] Unexpected columns in {path}: {df.columns}")
        frames.append(df[["Tier1 Tech", "EF (kgCo2/MWh)"]].copy())

    ef_all = pd.concat(frames, axis=0)

    ef_all = ef_all.groupby("Tier1 Tech", as_index=False)["EF (kgCo2/MWh)"].mean()
    ef_all["EF_kg_per_kwh"] = ef_all["EF (kgCo2/MWh)"] / 1000.0

    ef_map = dict(zip(ef_all["Tier1 Tech"], ef_all["EF_kg_per_kwh"]))
    print("\n[AEF-1A] EF map (kgCO2/kWh):")
    for k, v in ef_map.items():
        print("  ", k, "->", f"{v:.4f}")
    return ef_map

ef_map = load_ef_map(EMBER_CO2_DIR)


# ---------- Helper: compute CO2 and AEF from actual gen ----------

def add_co2_and_aef_from_gen(df: pd.DataFrame,
                             ef_map: dict,
                             name: str) -> pd.DataFrame:
    df = df.copy()
    eps = 1e-6

    # Determine which gen_* columns are in both df and EF map
    gen_cols_from_ef = [tech for tech in ef_map.keys() if tech in df.columns]
    if not gen_cols_from_ef:
        raise RuntimeError(f"[AEF-1A] No overlapping gen_* columns + EF map for {name}")

    print(f"\n[AEF-1A] {name}: using gen techs:", gen_cols_from_ef)

    # Compute per-tech CO2 (kg/h)
    co2_cols = []
    for tech in gen_cols_from_ef:
        ef_kg_per_kwh = ef_map[tech]
        out_col = f"co2_{tech}_kgph"
        df[out_col] = df[tech] * 1000.0 * ef_kg_per_kwh
        co2_cols.append(out_col)

    df["co2_total_kgph"] = df[co2_cols].sum(axis=1)

    # Total generation in kWh from the same techs
    df["gen_total_kwh"] = df[gen_cols_from_ef].sum(axis=1) * 1000.0

    # AEF as kgCO2 per kWh consumed/produced
    df["aef_true_kg_per_kwh"] = df["co2_total_kgph"] / (df["gen_total_kwh"] + eps)

    # Basic debug stats
    print(f"[AEF-1A] {name} co2_total_kgph stats:")
    print(df["co2_total_kgph"].describe())
    print(f"\n[AEF-1A] {name} AEF_true stats (kgCO2/kWh):")
    print(df["aef_true_kg_per_kwh"].describe())

    return df

train_with_aef = add_co2_and_aef_from_gen(train_master, ef_map, "TRAIN(2022–2023)")
test_with_aef  = add_co2_and_aef_from_gen(test_master,  ef_map, "TEST(2024)")


# ---------- Build AEF-ML training tables (features + target) ----------

aef_feature_cols = [
    "da_total_load_mw",
    "da_residual_load_mw",
    "da_net_imports_mw",
    "da_import_share",
    "da_elec_price_EURperMWh",
    "da_gas_price_migbasPVB_EURperMWh",
    "da_res_share_wind",
    "da_res_share_solar",
    "hour",
    "dow",
    "month",
    "is_weekend",
    "is_holiday",
]

missing_train = [c for c in aef_feature_cols if c not in train_with_aef.columns]
missing_test  = [c for c in aef_feature_cols if c not in test_with_aef.columns]
if missing_train or missing_test:
    raise ValueError(f"[AEF-1A] Missing features – train: {missing_train}, test: {missing_test}")

aef_target_col = "aef_true_kg_per_kwh"

aef_train_tbl = train_with_aef[aef_feature_cols + [aef_target_col]].copy()
aef_test_tbl  = test_with_aef[aef_feature_cols + [aef_target_col]].copy()

aef_train_path = os.path.join(TIER2_AEF_DIR, "tier2_1_aef_train_2022_2023.csv")
aef_test_path  = os.path.join(TIER2_AEF_DIR, "tier2_1_aef_true_2024.csv")

aef_train_tbl.reset_index().to_csv(aef_train_path, index=False)
aef_test_tbl.reset_index().to_csv(aef_test_path, index=False)

print(f"\n[AEF-1A] Saved AEF train table -> {aef_train_path} (rows={len(aef_train_tbl)})")
print(f"[AEF-1A] Saved AEF TRUE 2024 table -> {aef_test_path} (rows={len(aef_test_tbl)})")


# ---------- Quick sanity plots ----------

try:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    train_with_aef["aef_true_kg_per_kwh"].hist(bins=40, ax=axes[0], alpha=0.7)
    axes[0].set_title("AEF_true distribution – 2022–2023")
    axes[0].set_xlabel("kgCO2/kWh")
    axes[0].set_ylabel("Hours")

    test_with_aef["aef_true_kg_per_kwh"].hist(bins=40, ax=axes[1], alpha=0.7)
    axes[1].set_title("AEF_true distribution – 2024")
    axes[1].set_xlabel("kgCO2/kWh")
    axes[1].set_ylabel("Hours")

    plt.tight_layout()
    hist_path = os.path.join(TIER2_AEF_DIR, "tier2_1_aef_true_hist_2022_2024.png")
    plt.savefig(hist_path, dpi=150)
    plt.close(fig)
    print("[AEF-1A] Saved AEF histograms ->", hist_path)
except Exception as e:
    print("[AEF-1A] Skipped AEF plots due to error:", e)

print("\n[AEF-1A] Done. Next cell will build AEF_2024 from Tier-1 forecasts.")


## 2.2 AEF from Tier-1 forecasted generation


In [ ]:
# =========================
# Tier 2.1.B – AEF 2024 from Tier-1 forecasts (fixed tz alignment)
# =========================

import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TIER2_DIR      = str(TIER2_WORK)
TIER2_AEF_DIR  = os.path.join(TIER2_DIR, "tier2_1_aef")
DATA_PROC_DIR  = str(TIER1_WORK)
EMBER_CO2_DIR  = str(EMBER_SOURCE)

TRUE_2024_CSV  = os.path.join(TIER2_AEF_DIR, "tier2_1_aef_true_2024.csv")
OUT_FORECAST_CSV = os.path.join(TIER2_AEF_DIR, "tier2_1_aef_forecast_from_tier1_2024_full.csv")
OUT_METRICS_CSV  = os.path.join(TIER2_AEF_DIR, "tier2_1_aef_eval_tier1_vs_true_2024_metrics.csv")
OUT_SERIES_CSV   = os.path.join(TIER2_AEF_DIR, "tier2_1_aef_eval_tier1_vs_true_2024_series.csv")

print("[AEF-1B] Loading TRUE AEF 2024 table...")
aef_true_2024 = (
    pd.read_csv(TRUE_2024_CSV, parse_dates=["datetime_utc"])
      .set_index("datetime_utc")
      .sort_index()
)

# ensure UTC tz-aware index
if aef_true_2024.index.tz is None:
    aef_true_2024.index = aef_true_2024.index.tz_localize("UTC")
else:
    aef_true_2024 = aef_true_2024.tz_convert("UTC")

print("[AEF-1B] TRUE AEF 2024:", aef_true_2024.shape,
      "| range:", aef_true_2024.index.min(), "->", aef_true_2024.index.max())

# --------- EF map from EMBER files (2022–2024) ----------

ef_files = [
    os.path.join(EMBER_CO2_DIR, f)
    for f in os.listdir(EMBER_CO2_DIR)
    if f.startswith("gv2-tier2-ember-per-tech-emission-") and f.endswith(".csv")
]
if not ef_files:
    raise RuntimeError("[AEF-1B] No EF files found in EMBER_CO2_DIR.")

ef_frames = []
for path in sorted(ef_files):
    df_ef = pd.read_csv(path)
    if "Tier1 Tech" not in df_ef.columns or "EF (kgCo2/MWh)" not in df_ef.columns:
        raise ValueError(f"[EF] Unexpected columns in {path}: {df_ef.columns}")
    df_ef = df_ef[["Tier1 Tech", "EF (kgCo2/MWh)"]].copy()
    df_ef["EF_kg_per_kwh"] = df_ef["EF (kgCo2/MWh)"] / 1000.0
    ef_frames.append(df_ef)

ef_df = (pd.concat(ef_frames, axis=0)
           .drop_duplicates(subset=["Tier1 Tech"])
           .set_index("Tier1 Tech")
           .sort_index())
ef_map = ef_df["EF_kg_per_kwh"].to_dict()

print("\n[AEF-1B] EF map (kgCO2/kWh):")
for k, v in ef_map.items():
    print("  ", k, "->", f"{v:.4f}")

tech_cols = sorted(ef_map.keys())

# --------- Load Tier-1 per-tech predictions and align tz ----------

print("\n[AEF-1B] Loading Tier-1 per-tech hourly predictions for 2024...")

pred_frames = []

TIER1_FLAT_DIR = os.path.join(DATA_PROC_DIR, "tier1forecasts")

for tech in tech_cols:

    matches = []

    tech_folder = os.path.join(DATA_PROC_DIR, tech)
    pattern_sub = os.path.join(tech_folder, "*hourly_predictions_2024*.csv")
    matches = prediction_paths(tech)


    if not matches and os.path.isdir(TIER1_FLAT_DIR):
        pattern_flat = os.path.join(
            TIER1_FLAT_DIR,
            f"*{tech}*hourly_predictions_2024*.csv"
        )
        matches = glob.glob(pattern_flat)

    if not matches:
        print(f"[AEF-1B] WARNING: no prediction file for {tech} in subfolder or tier1forecasts – skipping.")
        continue

    pred_path = matches[0]

    df_pred = (
        pd.read_csv(pred_path, parse_dates=["datetime_utc"])
          .set_index("datetime_utc")
          .sort_index()
    )


    if df_pred.index.tz is None:
        df_pred.index = df_pred.index.tz_localize("UTC")
    else:
        df_pred = df_pred.tz_convert("UTC")

    # fossil gas special case, if u run code after tier 1 wihtour edit in the csv file or master file
    #if tech == "gen_fossil_gas_mw":
     #   pred_col = "y_pred"
    #else:
    pred_col = "y_pred_mw"

    df_pred = df_pred.rename(columns={"y_pred": "y_pred_mw", "y_true": "y_true_mw"})
    if pred_col not in df_pred.columns:
        raise ValueError(
            f"[AEF-1B] Column '{pred_col}' not found in {pred_path} "
            f"(cols={df_pred.columns.tolist()})"
        )

    df_pred = df_pred[[pred_col]].rename(columns={pred_col: tech})
    pred_frames.append(df_pred)

    print(f"[AEF-1B] {tech}: loaded {df_pred.shape[0]} rows from {os.path.basename(pred_path)}")

if not pred_frames:
    raise RuntimeError("[AEF-1B] No Tier-1 prediction series loaded.")

pred_all = pd.concat(pred_frames, axis=1).sort_index()

# align prediction horizon to TRUE AEF horizon
pred_all = pred_all.reindex(aef_true_2024.index)

print("\n[AEF-1B] Combined Tier-1 prediction matrix:", pred_all.shape)
print("[AEF-1B] 2024_Tier1 – raw prediction ranges (MW):")
stats = pred_all.agg(['min', 'max', 'mean']).T
print(stats)


# sanity check: ensure we did not kill everything by tz mismatch
if pred_all.isna().all().all():
    raise RuntimeError(
        "[AEF-1B] All Tier-1 predictions are NaN after reindexing. "
        "Check datetime_utc formats / timezones in prediction CSVs."
    )

# --------- Helper: compute AEF from per-tech MW predictions ----------

def compute_aef_from_pred(df_pred: pd.DataFrame, ef_map: dict, label: str):
    df = df_pred.copy()
    eps = 1e-6

    co2_cols = []
    for tech, ef in ef_map.items():
        if tech not in df.columns:
            continue
        out_col = f"co2_{tech}_kgph_pred"
        df[out_col] = df[tech] * 1000.0 * ef   # MW → kW → kg
        co2_cols.append(out_col)

    if not co2_cols:
        df["co2_total_kgph_pred"] = 0.0
    else:
        df["co2_total_kgph_pred"] = df[co2_cols].sum(axis=1)

    df["gen_total_mw_pred"]  = df[[c for c in df.columns if c.startswith("gen_")]].sum(axis=1)
    df["gen_total_kwh_pred"] = df["gen_total_mw_pred"] * 1000.0
    df["aef_tier1_forecast_kg_per_kwh"] = df["co2_total_kgph_pred"] / (df["gen_total_kwh_pred"] + eps)

    print(f"\n[AEF-1B] {label} co2_total_kgph_pred stats:")
    print(df["co2_total_kgph_pred"].describe())
    print(f"\n[AEF-1B] {label} AEF_tier1_forecast stats (kgCO2/kWh):")
    print(df["aef_tier1_forecast_kg_per_kwh"].describe())

    return df

# --------- Compute Tier-1 based AEF forecast and compare to TRUE 2024 ----------

pred_with_aef = compute_aef_from_pred(pred_all, ef_map, "2024_Tier1")

out = pd.concat(
    [
        aef_true_2024[["aef_true_kg_per_kwh"]],
        pred_with_aef[["aef_tier1_forecast_kg_per_kwh"]],
    ],
    axis=1,
)
out = out.dropna(subset=["aef_true_kg_per_kwh", "aef_tier1_forecast_kg_per_kwh"]).copy()

# metrics
y_true = out["aef_true_kg_per_kwh"].values
y_pred = out["aef_tier1_forecast_kg_per_kwh"].values

mae  = np.mean(np.abs(y_pred - y_true))
rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))
mape = np.mean(np.abs((y_pred - y_true) / np.maximum(np.abs(y_true), 1e-6))) * 100.0

metrics_df = pd.DataFrame(
    {
        "MAE_kg_per_kwh":  [mae],
        "RMSE_kg_per_kwh": [rmse],
        "MAPE_percent":    [mape],
        "n_hours":         [len(out)],
    }
)

print("\n[AEF-1B] AEF 2024 – Tier1 vs TRUE metrics:")
print(metrics_df)

# save outputs
pred_with_aef.reset_index().to_csv(OUT_FORECAST_CSV, index=False)
metrics_df.to_csv(OUT_METRICS_CSV, index=False)
out.reset_index().to_csv(OUT_SERIES_CSV, index=False)

print(f"[AEF-1B] Saved Tier-1-based AEF forecast table -> {OUT_FORECAST_CSV}")
print(f"[AEF-1B] Saved metrics -> {OUT_METRICS_CSV}")
print(f"[AEF-1B] Saved TRUE vs Tier1 AEF series -> {OUT_SERIES_CSV}")

# quick scatter + time series for sanity
try:
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(
        out["aef_true_kg_per_kwh"],
        out["aef_tier1_forecast_kg_per_kwh"],
        alpha=0.3,
        s=8,
        label="hours",
    )
    lims = [
        min(out["aef_true_kg_per_kwh"].min(), out["aef_tier1_forecast_kg_per_kwh"].min()),
        max(out["aef_true_kg_per_kwh"].max(), out["aef_tier1_forecast_kg_per_kwh"].max()),
    ]
    ax.plot(lims, lims, "k--", linewidth=1)
    ax.set_xlabel("AEF TRUE [kgCO2/kWh]")
    ax.set_ylabel("AEF Tier1 forecast [kgCO2/kWh]")
    ax.set_title("AEF 2024 – TRUE vs Tier1 forecast")
    plt.tight_layout()
    scatter_path = os.path.join(TIER2_AEF_DIR, "tier2_1_aef_true_vs_tier1_scatter_2024.png")
    plt.savefig(scatter_path, dpi=150)
    plt.close(fig)
    print(f"[AEF-1B] Saved scatter plot -> {scatter_path}")

    fig, ax = plt.subplots(figsize=(10, 4))
    out["aef_true_kg_per_kwh"].plot(ax=ax, label="TRUE", linewidth=1)
    out["aef_tier1_forecast_kg_per_kwh"].plot(ax=ax, label="Tier1", linewidth=1, alpha=0.8)
    ax.set_ylabel("kgCO2/kWh")
    ax.set_title("AEF 2024 – TRUE vs Tier1 forecast (time series)")
    ax.legend()
    plt.tight_layout()
    ts_path = os.path.join(TIER2_AEF_DIR, "tier2_1_aef_true_vs_tier1_timeseries_2024.png")
    plt.savefig(ts_path, dpi=150)
    plt.close(fig)
    print(f"[AEF-1B] Saved time-series plot -> {ts_path}")
except Exception as e:
    print("[AEF-1B] Skipped plots due to error:", e)

print("\n[AEF-1B] Done. Next step will be Tier 2.1.C – ML AEF model for 2022–23 and comparison on 2024.")


## 2.3 UpRamp MEF proxy


In [ ]:
# UpRamp MEF: original ramp horizons, train-calibrated shrinkage thresholds,
# technology-factor mappings and actual/forecast evaluation. Paths are configured below.

import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Plot style (your settings)
# -----------------------------
def apply_plot_settings(ax, title, xlabel, ylabel):
    ax.set_title(title, fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel(xlabel, fontsize=12, labelpad=10)
    ax.set_ylabel(ylabel, fontsize=12, labelpad=10)
    ax.tick_params(axis='both', which='major', labelsize=10)
    for spine in ax.spines.values():
        spine.set_color('#dadedf')
        spine.set_linewidth(2)
    ax.grid(False)

def savefig_png_svg(fig, out_noext, dpi=150):
    png_path = out_noext + ".png"
    svg_path = out_noext + ".svg"
    fig.savefig(png_path, dpi=dpi, bbox_inches="tight")
    fig.savefig(svg_path, bbox_inches="tight")
    plt.close(fig)
    return png_path, svg_path

# -----------------------------
# CONFIG
# -----------------------------
BASE_DIR = str(TIER2_WORK)
GENMIX_DIR = str(TIER1_WORK)
EMBER_DIR = str(EMBER_SOURCE)
AEF_DIR = str(TIER2_WORK / "tier2_1_aef")

OUT_DIR    = os.path.join(BASE_DIR, "tier2_2_mef_vnew2")
os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_GENMIX_FILE = str(TIER1_WORK / "master_ES_train_2022_2023_interp.csv")

EMBER_2022 = os.path.join(EMBER_DIR, "gv2-tier2-ember-per-tech-emission-2022.csv")
EMBER_2023 = os.path.join(EMBER_DIR, "gv2-tier2-ember-per-tech-emission-2023.csv")
EMBER_2024 = os.path.join(EMBER_DIR, "gv2-tier2-ember-per-tech-emission-2024.csv")

AEF_SERIES_FILE = os.path.join(AEF_DIR, "tier2_1_aef_eval_tier1_vs_true_2024_series.csv")

# Sweeps
K_LIST = [1, 3]
Q_LIST = [0.10, 0.20]

# Small eps
EPS_STACK = 1e-9

# -----------------------------
# Helper: UTC index alignment
# -----------------------------
def ensure_utc_index(df: pd.DataFrame, dt_col: str = "datetime_utc") -> pd.DataFrame:
    df = df.copy()
    if dt_col in df.columns:
        df[dt_col] = pd.to_datetime(df[dt_col], utc=True)
        df = df.set_index(dt_col)
    else:
        df.index = pd.to_datetime(df.index, utc=True)
    df = df.sort_index()
    if df.index.tz is None:
        df.index = df.index.tz_localize("UTC")
    else:
        df.index = df.index.tz_convert("UTC")
    return df

# -----------------------------
# Helper: Load Ember EF map (kgCO2/kWh)
# -----------------------------
def load_ember_ef_map(paths, label):
    frames = []
    for p in paths:
        if not os.path.exists(p):
            raise FileNotFoundError(f"[VNEW2] Missing Ember EF file ({label}): {p}")
        df = pd.read_csv(p)
        required = ["Tier1 Tech", "EF (kgCo2/MWh)"]
        miss = [c for c in required if c not in df.columns]
        if miss:
            raise ValueError(f"[VNEW2] Ember EF file missing columns {miss}: {p}")
        df = df[["Tier1 Tech", "EF (kgCo2/MWh)"]].copy()
        df["EF_kg_per_kwh"] = df["EF (kgCo2/MWh)"] / 1000.0
        frames.append(df[["Tier1 Tech", "EF_kg_per_kwh"]])
    ef_all = pd.concat(frames, axis=0)
    ef_avg = ef_all.groupby("Tier1 Tech", as_index=False)["EF_kg_per_kwh"].mean()
    ef_map = dict(zip(ef_avg["Tier1 Tech"], ef_avg["EF_kg_per_kwh"]))
    return ef_map, ef_avg

# -----------------------------
# Helper: Compute k-hour ramp MEF_up proxy + stack_up
#   ΔP_i^(k)(t) = P_i(t) - P_i(t-k)
#   ΔP_i^up     = max(ΔP_i^(k), 0)
#   stack_up    = Σ ΔP_i^up
#   CO2_up      = Σ ΔP_i^up * EF_i
#   MEF_up      = CO2_up / stack_up  if stack_up>0 else NaN
# -----------------------------
def compute_mef_up_proxy_k(gen_df: pd.DataFrame, ef_map: dict, tech_cols: list, k: int, label: str) -> pd.DataFrame:
    df = gen_df[tech_cols].copy().sort_index()
    for c in tech_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    dP = df.diff(k)
    dP_up = dP.clip(lower=0.0)

    stack_up = dP_up.sum(axis=1)

    ef_vec = np.array([ef_map[t] for t in tech_cols], dtype=float)
    co2_up = (dP_up.values * ef_vec).sum(axis=1)

    mef_up = np.full(shape=(len(stack_up),), fill_value=np.nan, dtype=float)
    mask = stack_up.values > EPS_STACK
    mef_up[mask] = co2_up[mask] / stack_up.values[mask]

    out = pd.DataFrame(index=df.index)
    out[f"{label}_stack_up_mw_k{k}"] = stack_up
    out[f"{label}_co2_up_weighted_k{k}"] = co2_up
    out[f"{label}_mef_up_kg_per_kwh_k{k}"] = mef_up
    return out

# -----------------------------
# Helper: Compute tau from TRAIN stack_up distribution
# -----------------------------
def compute_tau_from_train(stack_up: pd.Series, q: float) -> float:
    s = stack_up.dropna()
    s = s[s > EPS_STACK]
    if len(s) == 0:
        return np.nan
    return float(s.quantile(q))

# -----------------------------
# Helper: Shrinkage-to-AEF stabilization
#   w(t) = stack_up(t) / (stack_up(t) + tau)
#   MEF*(t) = w(t)*MEF_up(t) + (1-w(t))*AEF(t)
# -----------------------------
def stabilize_mef(mef_up: pd.Series, stack_up: pd.Series, aef: pd.Series, tau: float) -> pd.Series:
    if (tau is None) or (not np.isfinite(tau)) or (tau <= 0):
        return mef_up.copy()
    w = stack_up / (stack_up + tau)
    mef_star = w * mef_up + (1.0 - w) * aef
    return mef_star

# -----------------------------
# Weight diagnostics (hourly + summary)
# -----------------------------
def compute_weight_diagnostics_tables(
    index_utc,
    tau_mw: float,
    stack_up: pd.Series,
    mef_raw: pd.Series,
    aef: pd.Series,
    mef_final: pd.Series,
    tag: str,
):
    df = pd.DataFrame(index=index_utc)

    stack_up = pd.to_numeric(stack_up, errors="coerce")
    mef_raw  = pd.to_numeric(mef_raw,  errors="coerce")
    aef      = pd.to_numeric(aef,      errors="coerce")
    mef_final= pd.to_numeric(mef_final,errors="coerce")

    # w(t)=stack/(stack+tau)
    if (tau_mw is None) or (not np.isfinite(tau_mw)) or (tau_mw <= 0):
        w = pd.Series(np.nan, index=index_utc)
    else:
        w = stack_up / (stack_up + float(tau_mw))
        w = w.clip(lower=0.0, upper=1.0)

    mef_component = w * mef_raw
    aef_component = (1.0 - w) * aef
    recon_err = (mef_component + aef_component) - mef_final

    df["stack_up_mw"] = stack_up
    df["tau_mw"] = float(tau_mw) if np.isfinite(tau_mw) else np.nan
    df["w_mef"] = w
    df["mef_raw_kg_per_kwh"] = mef_raw
    df["aef_kg_per_kwh"] = aef
    df["mef_final_kg_per_kwh"] = mef_final
    df["mef_component"] = mef_component
    df["aef_component"] = aef_component
    df["reconstruction_error"] = recon_err

    w_valid = w.dropna()
    stack_valid = stack_up.dropna()

    thresholds = [0.2, 0.5, 0.8, 0.9]
    summary = {
        "tag": tag,
        "tau_mw": float(tau_mw) if np.isfinite(tau_mw) else np.nan,
        "n_hours_total": int(len(df)),
        "n_hours_w_valid": int(len(w_valid)),
        "w_mean": float(w_valid.mean()) if len(w_valid) else np.nan,
        "w_median": float(w_valid.median()) if len(w_valid) else np.nan,
        "w_q10": float(w_valid.quantile(0.10)) if len(w_valid) else np.nan,
        "w_q90": float(w_valid.quantile(0.90)) if len(w_valid) else np.nan,
        "stack_mean_mw": float(stack_valid.mean()) if len(stack_valid) else np.nan,
        "stack_median_mw": float(stack_valid.median()) if len(stack_valid) else np.nan,
        "stack_q10_mw": float(stack_valid.quantile(0.10)) if len(stack_valid) else np.nan,
        "stack_q90_mw": float(stack_valid.quantile(0.90)) if len(stack_valid) else np.nan,
        "recon_err_abs_mean": float(np.nanmean(np.abs(recon_err.values))),
        "recon_err_abs_max": float(np.nanmax(np.abs(recon_err.values))),
    }
    for th in thresholds:
        summary[f"frac_w_ge_{th}"] = float((w_valid >= th).mean()) if len(w_valid) else np.nan

    summary_df = pd.DataFrame([summary])
    return df, summary_df

# ============================================================
# 0) Load AEF series (2024)
# ============================================================
if not os.path.exists(AEF_SERIES_FILE):
    raise FileNotFoundError(f"[VNEW2] Missing AEF series file: {AEF_SERIES_FILE}")

aef_2024 = pd.read_csv(AEF_SERIES_FILE)
needed_aef = ["datetime_utc", "aef_true_kg_per_kwh", "aef_tier1_forecast_kg_per_kwh"]
miss = [c for c in needed_aef if c not in aef_2024.columns]
if miss:
    raise ValueError(f"[VNEW2] AEF series missing columns {miss}. Found: {aef_2024.columns.tolist()}")

aef_2024 = ensure_utc_index(aef_2024, "datetime_utc")
aef_2024 = aef_2024[["aef_true_kg_per_kwh", "aef_tier1_forecast_kg_per_kwh"]].copy()

# ============================================================
# 1) Load EF maps
# ============================================================
ef_train_map, ef_train_df = load_ember_ef_map([EMBER_2022, EMBER_2023], label="2022-2023")
ef_2024_map,  ef_2024_df  = load_ember_ef_map([EMBER_2024], label="2024")

# ============================================================
# 2) Load TRAIN genmix (actual, 2022–2023)
# ============================================================
if not os.path.exists(TRAIN_GENMIX_FILE):
    raise FileNotFoundError(f"[VNEW2] Missing TRAIN genmix file: {TRAIN_GENMIX_FILE}")

train_gen = pd.read_csv(TRAIN_GENMIX_FILE)
train_gen = ensure_utc_index(train_gen, "datetime_utc")

train_techs = sorted([t for t in ef_train_map.keys() if t in train_gen.columns])
if not train_techs:
    raise RuntimeError("[VNEW2] No overlapping gen tech columns between TRAIN genmix file and TRAIN EF map.")

# ============================================================
# 3) Load TEST 2024 per-tech prediction files
# ============================================================
if not os.path.isdir(GENMIX_DIR):
    raise FileNotFoundError(f"[VNEW2] Missing folder: {GENMIX_DIR}")

techs_2024 = sorted(list(ef_2024_map.keys()))

true_frames = []
pred_frames = []
loaded_techs = []

for tech in techs_2024:
    pattern = os.path.join(GENMIX_DIR, f"{tech}*hourly_predictions_2024*.csv")
    matches = prediction_paths(tech)
    if not matches:
        print(f"[VNEW2] WARNING: missing 2024 genmix file for {tech} (pattern: {pattern})")
        continue

    path = matches[0]
    df = pd.read_csv(path).rename(columns={"y_true": "y_true_mw", "y_pred": "y_pred_mw"})
    needed = ["datetime_utc", "y_true_mw", "y_pred_mw"]
    miss = [c for c in needed if c not in df.columns]
    if miss:
        raise ValueError(f"[VNEW2] {os.path.basename(path)} missing {miss}. Found: {df.columns.tolist()}")

    df = ensure_utc_index(df, "datetime_utc")
    df["y_true_mw"] = pd.to_numeric(df["y_true_mw"], errors="coerce")
    df["y_pred_mw"] = pd.to_numeric(df["y_pred_mw"], errors="coerce")

    true_frames.append(df[["y_true_mw"]].rename(columns={"y_true_mw": tech}))
    pred_frames.append(df[["y_pred_mw"]].rename(columns={"y_pred_mw": tech}))
    loaded_techs.append(tech)

if not true_frames or not pred_frames:
    raise RuntimeError("[VNEW2] No 2024 per-tech prediction files loaded. Check file names in Genmix_input.")

gen_true_2024 = pd.concat(true_frames, axis=1).sort_index()
gen_pred_2024 = pd.concat(pred_frames, axis=1).sort_index()

common_idx_2024 = gen_true_2024.index.intersection(gen_pred_2024.index)
gen_true_2024 = gen_true_2024.reindex(common_idx_2024)
gen_pred_2024 = gen_pred_2024.reindex(common_idx_2024)

aef_2024 = aef_2024.reindex(common_idx_2024)

test_techs = sorted([t for t in gen_true_2024.columns if t in ef_2024_map])
if not test_techs:
    raise RuntimeError("[VNEW2] No overlapping gen tech columns between 2024 prediction files and 2024 EF map.")

# ============================================================
# 4) Run the grid of (k,q)
# ============================================================
all_outputs = []
summary_rows = []

# A) TRAIN precompute
train_mef_by_k = {}
for k in K_LIST:
    train_mef_by_k[k] = compute_mef_up_proxy_k(
        gen_df=train_gen,
        ef_map=ef_train_map,
        tech_cols=train_techs,
        k=k,
        label="train_true"
    )

# B) Tau table
tau_table_rows = []
for k in K_LIST:
    stack_series = train_mef_by_k[k][f"train_true_stack_up_mw_k{k}"]
    for q in Q_LIST:
        tau = compute_tau_from_train(stack_series, q)
        tau_table_rows.append({
            "k": k, "q": q, "tau_mw": tau,
            "train_stack_up_pos_count": int((stack_series > EPS_STACK).sum())
        })

tau_table = pd.DataFrame(tau_table_rows)
tau_csv = os.path.join(OUT_DIR, "vnew2_train_tau_values_by_k_q.csv")
tau_table.to_csv(tau_csv, index=False)
all_outputs.append(tau_csv)

# TRAIN stack hist plots
for k in K_LIST:
    stack_series = train_mef_by_k[k][f"train_true_stack_up_mw_k{k}"].dropna()
    pos = stack_series[stack_series > EPS_STACK]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(pos.values, bins=60, alpha=0.8)
    for q in Q_LIST:
        tau = float(tau_table[(tau_table["k"] == k) & (tau_table["q"] == q)]["tau_mw"].iloc[0])
        ax.axvline(tau, linewidth=2, linestyle="--", label=f"q={int(q*100)}% → tau={tau:.2f} MW")
    apply_plot_settings(
        ax,
        title=f"TRAIN (2022–2023) stack_up distribution (k={k}h) with tau quantiles",
        xlabel="stack_up (MW)",
        ylabel="count (hours)"
    )
    ax.legend(fontsize=10)
    out_noext = os.path.join(OUT_DIR, f"vnew2_train_stackup_hist_k{k}")
    png, svg = savefig_png_svg(fig, out_noext)
    all_outputs += [png, svg]

# TRAIN raw stats
train_stats_rows = []
for k in K_LIST:
    mef_series = train_mef_by_k[k][f"train_true_mef_up_kg_per_kwh_k{k}"]
    stack_series = train_mef_by_k[k][f"train_true_stack_up_mw_k{k}"]
    train_stats_rows.append({
        "k": k,
        "n_hours": int(len(mef_series)),
        "n_nan_mef": int(mef_series.isna().sum()),
        "nan_frac_mef": float(mef_series.isna().mean()),
        "n_pos_stack": int((stack_series > EPS_STACK).sum()),
        "mef_mean": float(mef_series.dropna().mean()),
        "mef_std": float(mef_series.dropna().std()),
        "mef_p05": float(mef_series.dropna().quantile(0.05)) if mef_series.notna().any() else np.nan,
        "mef_p50": float(mef_series.dropna().quantile(0.50)) if mef_series.notna().any() else np.nan,
        "mef_p95": float(mef_series.dropna().quantile(0.95)) if mef_series.notna().any() else np.nan,
    })
train_stats = pd.DataFrame(train_stats_rows)
train_stats_csv = os.path.join(OUT_DIR, "vnew2_train_mef_stack_stats_by_k.csv")
train_stats.to_csv(train_stats_csv, index=False)
all_outputs.append(train_stats_csv)

# ============================================================
# 5) Per config outputs + NEW weight diagnostics CSVs
# ============================================================
def compute_metrics_like_aef(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae  = float(np.mean(np.abs(y_pred - y_true)))
    rmse = float(np.sqrt(np.mean((y_pred - y_true) ** 2)))
    mape = float(np.mean(np.abs((y_pred - y_true) / np.maximum(np.abs(y_true), 1e-6))) * 100.0)
    return mae, rmse, mape

def pick_week_window(index_utc):
    start = index_utc.min()
    end = start + pd.Timedelta(days=7)
    return start, end

#collect all weight-diag summaries in one file too
weight_summary_all = []

for k in K_LIST:
    out_true = compute_mef_up_proxy_k(gen_true_2024, ef_2024_map, test_techs, k=k, label="true")
    out_pred = compute_mef_up_proxy_k(gen_pred_2024, ef_2024_map, test_techs, k=k, label="pred")

    mef_true_raw = out_true[f"true_mef_up_kg_per_kwh_k{k}"]
    mef_pred_raw = out_pred[f"pred_mef_up_kg_per_kwh_k{k}"]
    stack_true   = out_true[f"true_stack_up_mw_k{k}"]
    stack_pred   = out_pred[f"pred_stack_up_mw_k{k}"]

    for q in Q_LIST:
        tag = f"k{k}_q{int(q*100)}"
        cfg_dir = os.path.join(OUT_DIR, f"vnew2_{tag}")
        os.makedirs(cfg_dir, exist_ok=True)

        tau = float(tau_table[(tau_table["k"] == k) & (tau_table["q"] == q)]["tau_mw"].iloc[0])

        aef_true_2024 = aef_2024["aef_true_kg_per_kwh"]
        aef_pred_2024 = aef_2024["aef_tier1_forecast_kg_per_kwh"]

        mef_true_star = stabilize_mef(mef_true_raw, stack_true, aef_true_2024, tau=tau)
        mef_pred_star = stabilize_mef(mef_pred_raw, stack_pred, aef_pred_2024, tau=tau)

        mef_true_final = mef_true_star.fillna(aef_true_2024)
        mef_pred_final = mef_pred_star.fillna(aef_pred_2024)

        # Build detailed output table
        df_out = pd.DataFrame(index=common_idx_2024)
        for tech in test_techs:
            df_out[f"{tech}_true_mw"] = gen_true_2024[tech]
            df_out[f"{tech}_pred_mw"] = gen_pred_2024[tech]

        df_out[f"stack_up_true_mw_k{k}"] = stack_true
        df_out[f"stack_up_pred_mw_k{k}"] = stack_pred
        df_out[f"mef_up_true_raw_kg_per_kwh_k{k}"] = mef_true_raw
        df_out[f"mef_up_pred_raw_kg_per_kwh_k{k}"] = mef_pred_raw

        df_out["aef_true_kg_per_kwh"] = aef_true_2024
        df_out["aef_tier1_forecast_kg_per_kwh"] = aef_pred_2024

        df_out[f"mef_true_star_shrink_kg_per_kwh_{tag}"] = mef_true_star
        df_out[f"mef_pred_star_shrink_kg_per_kwh_{tag}"] = mef_pred_star
        df_out[f"mef_true_final_filled_kg_per_kwh_{tag}"] = mef_true_final
        df_out[f"mef_pred_final_filled_kg_per_kwh_{tag}"] = mef_pred_final

        out_csv = os.path.join(cfg_dir, f"vnew2_mef_proxy_fulltable_2024_{tag}.csv")
        df_out.reset_index().rename(columns={"index": "datetime_utc"}).to_csv(out_csv, index=False)
        all_outputs.append(out_csv)

        # weight diagnostics (hourly + summary) for TRUE and PRED ----
        # TRUE weight diag
        wt_hourly, wt_summary = compute_weight_diagnostics_tables(
            index_utc=common_idx_2024,
            tau_mw=tau,
            stack_up=stack_true,
            mef_raw=mef_true_raw,
            aef=aef_true_2024,
            mef_final=mef_true_final,
            tag=f"{tag}_TRUE"
        )
        wt_hourly_csv = os.path.join(cfg_dir, f"vnew2_weight_diagnostics_hourly_2024_{tag}_TRUE.csv")
        wt_summary_csv = os.path.join(cfg_dir, f"vnew2_weight_diagnostics_summary_2024_{tag}_TRUE.csv")
        wt_hourly.reset_index().rename(columns={"index":"datetime_utc"}).to_csv(wt_hourly_csv, index=False)
        wt_summary.to_csv(wt_summary_csv, index=False)
        all_outputs += [wt_hourly_csv, wt_summary_csv]
        weight_summary_all.append(wt_summary.iloc[0].to_dict())

        # PRED weight diag
        wp_hourly, wp_summary = compute_weight_diagnostics_tables(
            index_utc=common_idx_2024,
            tau_mw=tau,
            stack_up=stack_pred,
            mef_raw=mef_pred_raw,
            aef=aef_pred_2024,
            mef_final=mef_pred_final,
            tag=f"{tag}_PRED"
        )
        wp_hourly_csv = os.path.join(cfg_dir, f"vnew2_weight_diagnostics_hourly_2024_{tag}_PRED.csv")
        wp_summary_csv = os.path.join(cfg_dir, f"vnew2_weight_diagnostics_summary_2024_{tag}_PRED.csv")
        wp_hourly.reset_index().rename(columns={"index":"datetime_utc"}).to_csv(wp_hourly_csv, index=False)
        wp_summary.to_csv(wp_summary_csv, index=False)
        all_outputs += [wp_hourly_csv, wp_summary_csv]
        weight_summary_all.append(wp_summary.iloc[0].to_dict())

        # Existing NaN diagnostics (kept)
        diag_rows = []
        def add_diag(name, s):
            s = pd.Series(s)
            diag_rows.append({
                "series": name,
                "n_hours": int(len(s)),
                "n_nan": int(s.isna().sum()),
                "nan_fraction": float(s.isna().mean()),
            })

        add_diag(f"mef_true_raw_{tag}", mef_true_raw)
        add_diag(f"mef_pred_raw_{tag}", mef_pred_raw)
        add_diag(f"mef_true_star_{tag}", mef_true_star)
        add_diag(f"mef_pred_star_{tag}", mef_pred_star)
        add_diag(f"mef_true_final_{tag}", mef_true_final)
        add_diag(f"mef_pred_final_{tag}", mef_pred_final)
        add_diag(f"stack_true_{tag}", stack_true)
        add_diag(f"stack_pred_{tag}", stack_pred)

        diag_df = pd.DataFrame(diag_rows)
        diag_csv = os.path.join(cfg_dir, f"vnew2_nan_and_weight_diagnostics_2024_{tag}.csv")
        diag_df.to_csv(diag_csv, index=False)
        all_outputs.append(diag_csv)

        # Metrics
        eval_df = pd.DataFrame({
            "true_final": mef_true_final,
            "pred_final": mef_pred_final
        }).dropna()

        mae, rmse, mape = compute_metrics_like_aef(eval_df["true_final"].values, eval_df["pred_final"].values)
        metrics_df = pd.DataFrame({
            "k": [k],
            "q": [q],
            "tau_mw": [tau],
            "MAE_kg_per_kwh": [mae],
            "RMSE_kg_per_kwh": [rmse],
            "MAPE_percent": [mape],
            "n_hours": [len(eval_df)],
        })
        metrics_csv = os.path.join(cfg_dir, f"vnew2_metrics_pred_vs_true_2024_{tag}.csv")
        metrics_df.to_csv(metrics_csv, index=False)
        all_outputs.append(metrics_csv)
        summary_rows.append(metrics_df.iloc[0].to_dict())

        # Distribution plot
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(mef_true_final.values, bins=60, alpha=0.65, label="TRUE (final)")
        ax.hist(mef_pred_final.values, bins=60, alpha=0.65, label="PRED (final)")
        apply_plot_settings(
            ax,
            title=f"MEF* distribution 2024 (final) – {tag} (tau={tau:.2f} MW)",
            xlabel="kgCO2/kWh",
            ylabel="count (hours)"
        )
        ax.legend(fontsize=10)
        out_noext = os.path.join(cfg_dir, f"vnew2_dist_mef_final_2024_{tag}")
        png, svg = savefig_png_svg(fig, out_noext)
        all_outputs += [png, svg]

        # Parity plot
        y_true = eval_df["true_final"].values
        y_pred = eval_df["pred_final"].values
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.scatter(y_true, y_pred, s=10, alpha=0.35, label="hours")
        lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
        ax.plot(lims, lims, "k--", linewidth=1.25, label="Ideal")
        apply_plot_settings(
            ax,
            title=f"MEF* parity 2024 – TRUE vs PRED ({tag})",
            xlabel="TRUE MEF* (final) [kgCO2/kWh]",
            ylabel="PRED MEF* (final) [kgCO2/kWh]"
        )
        ax.legend(fontsize=10)
        out_noext = os.path.join(cfg_dir, f"vnew2_parity_mef_final_2024_{tag}")
        png, svg = savefig_png_svg(fig, out_noext)
        all_outputs += [png, svg]

        # Full-year time series
        fig, ax = plt.subplots(figsize=(14, 4))
        pd.Series(mef_true_final, index=common_idx_2024).plot(ax=ax, linewidth=1.0, label=f"MEF* TRUE (final) {tag}")
        pd.Series(mef_pred_final, index=common_idx_2024).plot(ax=ax, linewidth=1.0, alpha=0.85, label=f"MEF* PRED (final) {tag}")
        aef_true_2024.plot(ax=ax, linewidth=0.9, alpha=0.55, label="AEF TRUE")
        aef_pred_2024.plot(ax=ax, linewidth=0.9, alpha=0.55, label="AEF Tier1 forecast")
        apply_plot_settings(
            ax,
            title=f"Full-year 2024 time series – MEF* + AEF ({tag})",
            xlabel="Time (UTC)",
            ylabel="kgCO2/kWh"
        )
        ax.legend(fontsize=9, ncol=2)
        out_noext = os.path.join(cfg_dir, f"vnew2_timeseries_full_year_2024_{tag}")
        png, svg = savefig_png_svg(fig, out_noext)
        all_outputs += [png, svg]

        # One-week plots
        week_start, week_end = pick_week_window(common_idx_2024)
        week_mask = (common_idx_2024 >= week_start) & (common_idx_2024 < week_end)

        week_true_mef = pd.Series(mef_true_final.values, index=common_idx_2024).loc[week_mask]
        week_pred_mef = pd.Series(mef_pred_final.values, index=common_idx_2024).loc[week_mask]
        week_aef_true = aef_true_2024.loc[week_mask]
        week_aef_pred = aef_pred_2024.loc[week_mask]

        fig, ax = plt.subplots(figsize=(14, 4))
        week_true_mef.plot(ax=ax, linewidth=1.2, label=f"MEF* TRUE (final) {tag}")
        week_pred_mef.plot(ax=ax, linewidth=1.2, alpha=0.9, label=f"MEF* PRED (final) {tag}")
        week_aef_true.plot(ax=ax, linewidth=1.0, alpha=0.55, label="AEF TRUE")
        week_aef_pred.plot(ax=ax, linewidth=1.0, alpha=0.55, label="AEF Tier1 forecast")
        apply_plot_settings(
            ax,
            title=f"One-week 2024 – MEF* + AEF ({tag}) [{week_start.date()} to {(week_end-pd.Timedelta(hours=1)).date()}]",
            xlabel="Time (UTC)",
            ylabel="kgCO2/kWh"
        )
        ax.legend(fontsize=9, ncol=2)
        out_noext = os.path.join(cfg_dir, f"vnew2_timeseries_one_week_overlay_2024_{tag}")
        png, svg = savefig_png_svg(fig, out_noext)
        all_outputs += [png, svg]

        fig, ax = plt.subplots(figsize=(14, 3.8))
        week_true_mef.plot(ax=ax, linewidth=1.2, label=f"MEF* TRUE (final) {tag}")
        week_aef_true.plot(ax=ax, linewidth=1.0, alpha=0.55, label="AEF TRUE")
        apply_plot_settings(
            ax,
            title=f"One-week 2024 – TRUE signals ({tag})",
            xlabel="Time (UTC)",
            ylabel="kgCO2/kWh"
        )
        ax.legend(fontsize=10)
        out_noext = os.path.join(cfg_dir, f"vnew2_timeseries_one_week_TRUE_2024_{tag}")
        png, svg = savefig_png_svg(fig, out_noext)
        all_outputs += [png, svg]

        fig, ax = plt.subplots(figsize=(14, 3.8))
        week_pred_mef.plot(ax=ax, linewidth=1.2, label=f"MEF* PRED (final) {tag}")
        week_aef_pred.plot(ax=ax, linewidth=1.0, alpha=0.55, label="AEF Tier1 forecast")
        apply_plot_settings(
            ax,
            title=f"One-week 2024 – PRED signals ({tag})",
            xlabel="Time (UTC)",
            ylabel="kgCO2/kWh"
        )
        ax.legend(fontsize=10)
        out_noext = os.path.join(cfg_dir, f"vnew2_timeseries_one_week_PRED_2024_{tag}")
        png, svg = savefig_png_svg(fig, out_noext)
        all_outputs += [png, svg]

        # Grid MEF file
        grid_mef = pd.DataFrame(index=common_idx_2024)
        grid_mef["datetime_utc"] = common_idx_2024
        grid_mef["grid_mef_pred_kg_per_kwh"] = mef_pred_final.values
        grid_mef["grid_mef_true_kg_per_kwh"] = mef_true_final.values
        grid_mef["aef_tier1_forecast_kg_per_kwh"] = aef_pred_2024.values
        grid_mef["aef_true_kg_per_kwh"] = aef_true_2024.values
        grid_mef["stack_up_pred_mw"] = stack_pred.values
        grid_mef["stack_up_true_mw"] = stack_true.values
        grid_mef["tau_mw_used"] = tau
        grid_mef["k_hours"] = k
        grid_mef["q_quantile"] = q

        grid_csv = os.path.join(cfg_dir, f"grid_mef_hourly_2024_{tag}.csv")
        grid_mef.to_csv(grid_csv, index=False)
        all_outputs.append(grid_csv)

# ============================================================
# 6) Global summary table and global weight summary
# ============================================================
summary_df = pd.DataFrame(summary_rows).sort_values(["k", "q"])
summary_csv = os.path.join(OUT_DIR, "vnew2_summary_metrics_all_configs.csv")
summary_df.to_csv(summary_csv, index=False)
all_outputs.append(summary_csv)

# weight summaries across all configs
weight_summary_all_df = pd.DataFrame(weight_summary_all)
weight_summary_all_csv = os.path.join(OUT_DIR, "vnew2_weight_diagnostics_summary_all_configs.csv")
weight_summary_all_df.to_csv(weight_summary_all_csv, index=False)
all_outputs.append(weight_summary_all_csv)

print("\n[VNEW2] Summary metrics across configs:")
print(summary_df)

print("\n[VNEW2] Weight diagnostic summaries across configs (TRUE+PRED):")
print(weight_summary_all_df)

print("UpRamp outputs:", OUT_DIR)


## 2.4 MSDR Soft MEF proxy


## MSDR input assembly

These tables arrange the existing actual and forecast technology columns for the rolling regression. The fossil-gas forecast column names are normalized on reading; their values are unchanged.


In [ ]:
# Assemble the genmix tables consumed by the original rolling MSDR block.
SOFT_INPUT = TIER2_WORK / "soft_input"
SOFT_INPUT.mkdir(parents=True, exist_ok=True)
gen_columns = [c for c in train_master.columns if c.startswith("gen_") and c in ef_map]
train_master.loc["2023-12-01":"2023-12-31", gen_columns].reset_index().to_csv(SOFT_INPUT / "genmix_actual_last month of 2023_tier2.3.csv", index=False)
test_master[gen_columns].reset_index().to_csv(SOFT_INPUT / "genmix_actual_2024_tier2.3.csv", index=False)
pred_all[gen_columns].reset_index().to_csv(SOFT_INPUT / "genmix_tier1forecast_2024_tier2.3.csv", index=False)


In [ ]:
# ============================================================
# MEF 2.3
# ------------------------------------------------------------
#
# A) MSDR outputs:
#    1) Smoothed regime probabilities exported per hour (and plotted)
#    2) Regime parameters exported (const, slope, sigma2)
#    3) Model-summary text dumps (NOT for every hour; selectable subset)
#
# B) DLR baseline block (rolling OLS on the same 168h window):
#    - slope/intercept/R2 per target hour
#    - timeseries plots for DLR slope
#    - overlay plot: DLR slope vs MSDR soft slope (same units)
#
# C) Regime interpretation helpers:
#    - fossil-share series from genmix (heuristic fossil tech detection)
#    - plots: fossil share vs MEF (daily) and fossil share vs regime probabilities (daily)
#    - correlation table between fossil share and regime probs / MEF
#
# D) No scaling:
#    - Slopes remain in physical units: kgCO2/MWh -> final kgCO2/kWh
#
# E) Stability
#    - robust target hour selection
#    - window conditioning checks (dg variance)
#    - corrected capping logic
#    - NaN/negative are explicitly reported (exact hours) and filled with AEF
#    - robust transition matrix reconstruction
#
# ============================================================

import os, glob, warnings, re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression

warnings.filterwarnings("ignore")

# ============================================================
# CONFIG
# ============================================================

# ---- I/O ----
INPUT_DIR = str(SOFT_INPUT)
OUT_DIR = str(TIER2_WORK / "mef2_3_v6_modified_outputs")

OUT_CSV_DIR  = os.path.join(OUT_DIR, "csv")
OUT_PLOT_DIR = os.path.join(OUT_DIR, "plots")
OUT_META_DIR = os.path.join(OUT_DIR, "meta")
OUT_TEXT_DIR = os.path.join(OUT_DIR, "model_summaries")
for d in [OUT_CSV_DIR, OUT_PLOT_DIR, OUT_META_DIR, OUT_TEXT_DIR]:
    os.makedirs(d, exist_ok=True)

APPLY_YEAR = 2024

ROLL_HOURS = 336
# ---- Selected two-regime configuration ----
K_LIST = [2]

# ============================================================
# Target-hour loop control (debug-fast)
# ============================================================
TARGET_HOURS_MODE = "full_year"  # full_year | first_n | random_n | explicit_list | range_slice
TARGET_FIRST_N     = 500
TARGET_RANDOM_N    = 200
TARGET_RANDOM_SEED = 42
TARGET_EXPLICIT_LIST = []
TARGET_RANGE_START = "2024-01-01 00:00:00+00:00"
TARGET_RANGE_END   = "2024-01-10 23:00:00+00:00"

# ============================================================
# Stability guards for ill-conditioned windows
# ============================================================
DG_STD_MIN = 1e-6
DG_EPS = 1e-6
DG_MIN_FRAC_BIG = 0.10  # require at least 10% of points with |dG| > DG_EPS

# ============================================================
# Capping (fixed logic)
# ============================================================
CAP_Q_LOW  = 0.001
CAP_Q_HIGH = 0.999
CAP_ABS_FACTOR = 2.0  # abs bound = ± factor * EF_MAX_2024

# ============================================================
# Smoothed vs filtered probabilities
# ============================================================
# Filtered:  P(S_t=k | y_1..y_t)
# Smoothed:  P(S_t=k | y_1..y_T)  (uses future data within the window)
PROB_MODE = "smoothed"  # use smoothed for outputs / soft+hard MEF selection

# ============================================================
# Model summary
# ============================================================
# Dumping summary for all hours is huge; use a controlled policy:
SUMMARY_DUMP_MODE = "some"  # none | some | all
SUMMARY_DUMP_MAX  = 20      # max windows to dump in "some" mode
SUMMARY_DUMP_EVERY = 50     # dump every Nth successful window in "some" mode

# ============================================================
# Plot style helper
# ============================================================
def apply_plot_settings(ax, title, xlabel, ylabel):
    ax.set_title(title, fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel(xlabel, fontsize=12, labelpad=10)
    ax.set_ylabel(ylabel, fontsize=12, labelpad=10)
    ax.tick_params(axis='both', which='major', labelsize=10)
    for spine in ax.spines.values():
        spine.set_color('#dadedf')
        spine.set_linewidth(2)
    ax.grid(False)

def save_fig(fig, filename_base):
    png_path = os.path.join(OUT_PLOT_DIR, f"{filename_base}.png")
    svg_path = os.path.join(OUT_PLOT_DIR, f"{filename_base}.svg")
    fig.savefig(png_path, dpi=150, bbox_inches="tight")
    fig.savefig(svg_path, bbox_inches="tight")
    plt.close(fig)

# ============================================================
# Utilities
# ============================================================
def _find_one(patterns, must_exist=True):
    matches = []
    for p in patterns:
        matches.extend(glob.glob(p))
    matches = sorted(list(set(matches)))
    if not matches and must_exist:
        raise FileNotFoundError(
            "No files matched patterns:\n" + "\n".join(patterns) +
            f"\n\nCheck INPUT_DIR={INPUT_DIR} and file names."
        )
    return matches[0] if matches else None

def read_any_table(path):
    if path.lower().endswith(".csv"):
        return pd.read_csv(path)
    if path.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(path)
    raise ValueError(f"Unsupported file type: {path}")

def to_utc_index_robust(df, col="datetime_utc"):
    df = df.copy()
    if col not in df.columns:
        raise ValueError(f"Missing datetime column '{col}'. Found: {list(df.columns)}")

    s = df[col].astype(str).str.strip()
    dt1 = pd.to_datetime(s, utc=True, errors="coerce")
    nat_rate = dt1.isna().mean()

    if nat_rate > 0.01:
        dt2 = pd.to_datetime(s, utc=True, errors="coerce", dayfirst=True)
        dt = dt2 if dt2.isna().mean() < nat_rate else dt1
    else:
        dt = dt1

    df[col] = dt
    df = df.dropna(subset=[col]).sort_values(col).set_index(col)

    if df.index.tz is None:
        df.index = df.index.tz_localize("UTC")
    else:
        df.index = df.index.tz_convert("UTC")

    df.index = df.index.floor("H")
    if df.index.has_duplicates:
        df = df.groupby(df.index).mean(numeric_only=True)

    return df.sort_index()

def complete_hourly_index(df, name_for_report):
    if df.empty:
        return df, []
    full_idx = pd.date_range(df.index.min(), df.index.max(), freq="H", tz="UTC")
    missing = full_idx.difference(df.index)
    pd.DataFrame({"datetime_utc_missing": missing}).to_csv(
        os.path.join(OUT_CSV_DIR, f"{name_for_report}_missing_hours.csv"),
        index=False
    )
    return df, list(missing)

def clip_negative_generation(df, gen_cols, label):
    neg_rows = []
    df2 = df.copy()
    for c in gen_cols:
        neg_mask = df2[c] < 0
        if neg_mask.any():
            ts_list = df2.index[neg_mask].tolist()
            neg_rows.append({
                "stream": label,
                "column": c,
                "n_negative": int(neg_mask.sum()),
                "first_negative_utc": str(ts_list[0]),
                "last_negative_utc": str(ts_list[-1]),
            })
            df2.loc[neg_mask, c] = 0.0
            pd.DataFrame({"datetime_utc": ts_list}).to_csv(
                os.path.join(OUT_CSV_DIR, f"negatives_{label}_{c}.csv"),
                index=False
            )
    pd.DataFrame(neg_rows).to_csv(
        os.path.join(OUT_CSV_DIR, f"negatives_summary_{label}.csv"),
        index=False
    )
    return df2

def load_ef(path, year_label):
    d = read_any_table(path).copy()
    cols_lower = {c.lower(): c for c in d.columns}
    tech_col = cols_lower.get("tier1 tech", None)

    ef_col = None
    for k, orig in cols_lower.items():
        if "ef" in k and ("kg" in k) and ("mwh" in k):
            ef_col = orig
            break

    if tech_col is None or ef_col is None:
        raise ValueError(
            f"EF file {year_label} missing expected columns. "
            f"Need 'Tier1 Tech' and EF column like 'EF (kgCo2/Mwh)'. Found: {list(d.columns)}"
        )

    d = d[[tech_col, ef_col]].rename(columns={tech_col: "tech", ef_col: "ef_kg_per_mwh"})
    d["tech"] = d["tech"].astype(str).str.strip()
    d["ef_kg_per_mwh"] = pd.to_numeric(d["ef_kg_per_mwh"], errors="coerce")
    d = d.dropna(subset=["tech", "ef_kg_per_mwh"])
    return d

def compute_emissions_one_row(gen_row, gen_cols, ef_map):
    s = 0.0
    missing = 0
    for c in gen_cols:
        ef = ef_map.get(c, np.nan)
        g  = gen_row.get(c, np.nan)
        if pd.isna(ef) or pd.isna(g):
            if pd.isna(ef):
                missing += 1
            continue
        s += float(g) * float(ef)
    return s, missing

# ============================================================
# Fossil-share helper (regime interpretation aid)
# ============================================================
def detect_fossil_cols(gen_cols):
    """
    Heuristic fossil tech detection from gen_ column names.
    Adjust if you want explicit mapping later.
    """
    fossil_keys = [
        "coal", "hard_coal", "oil",
        "gas", "fossil"
    ]
    exclude_keys = [
        "biomass", "waste", "renew", "solar", "wind", "hydro", "nuclear",
        "other"
    ]
    fossil = []
    for c in gen_cols:
        lc = c.lower()
        if any(k in lc for k in exclude_keys):
            continue
        if any(k in lc for k in fossil_keys):
            fossil.append(c)
    return sorted(list(set(fossil)))

def fossil_share_from_genmix(df_2024, gen_cols, label):
    fossil_cols = detect_fossil_cols(gen_cols)
    meta = {
        "label": label,
        "n_gen_cols": len(gen_cols),
        "fossil_cols_used": fossil_cols,
        "n_fossil_cols_used": len(fossil_cols),
    }
    with open(os.path.join(OUT_META_DIR, f"fossil_cols_{label}.json"), "w") as f:
        json.dump(meta, f, indent=2)

    if len(fossil_cols) == 0:
        # produce empty/NaN series for transparency
        s = pd.Series(index=df_2024.index, data=np.nan, name="fossil_share")
        s.to_frame().reset_index().rename(columns={"index":"datetime_utc"}).to_csv(
            os.path.join(OUT_CSV_DIR, f"fossil_share_{label}_hourly.csv"), index=False
        )
        return s, fossil_cols

    total = df_2024[gen_cols].sum(axis=1)
    fossil = df_2024[fossil_cols].sum(axis=1)
    share = (fossil / total).replace([np.inf, -np.inf], np.nan)
    share.name = "fossil_share"

    share.to_frame().reset_index().rename(columns={"index":"datetime_utc"}).to_csv(
        os.path.join(OUT_CSV_DIR, f"fossil_share_{label}_hourly.csv"), index=False
    )
    return share, fossil_cols

# ============================================================
# MSDR fitting helpers
# ============================================================
def safe_fit_msdr(dE, dG, K):
    endog = np.asarray(dE, dtype=float)
    exog  = np.asarray(dG, dtype=float).reshape(-1, 1)

    good = np.isfinite(endog) & np.isfinite(exog[:, 0])
    endog = endog[good]
    exog  = exog[good, :]

    if len(endog) < max(50, 10*K):
        return None, "too_few_obs"

    dg = exog[:, 0]
    if np.nanstd(dg) < DG_STD_MIN:
        return None, "dg_std_too_small"

    frac_big = float(np.mean(np.abs(dg) > DG_EPS))
    if frac_big < DG_MIN_FRAC_BIG:
        return None, "dg_near_zero_too_often"

    mod = MarkovRegression(
        endog=endog,
        exog=exog,
        k_regimes=K,
        trend="c",
        switching_exog=True,
        switching_trend=True,
        switching_variance=True
    )

    try:
        res = mod.fit(disp=False, maxiter=200)
        return res, "ok"
    except Exception:
        return None, "fit_failed"

def extract_slopes(res, K):
    params = np.asarray(res.params, dtype=float)
    names  = list(res.model.param_names)
    slopes = [np.nan]*K
    for nm, val in zip(names, params):
        n = nm.lower().replace(" ", "")
        if n.startswith("x1[") and "[" in nm and "]" in nm:
            try:
                k = int(nm.split("[")[-1].split("]")[0])
                if 0 <= k < K:
                    slopes[k] = float(val)
            except:
                pass
    slopes = np.asarray(slopes, dtype=float)
    if not np.isfinite(slopes).all():
        raise RuntimeError(f"Could not extract {K} slopes. got={slopes}")
    return slopes

def extract_intercepts_sigmas(res, K):
    params = np.asarray(res.params, dtype=float)
    names  = list(res.model.param_names)
    consts = [np.nan]*K
    sig2   = [np.nan]*K
    for nm, val in zip(names, params):
        n = nm.lower().replace(" ", "")
        if n.startswith("const["):
            try:
                k = int(nm.split("[")[-1].split("]")[0])
                if 0 <= k < K:
                    consts[k] = float(val)
            except:
                pass
        if "sigma2[" in n:
            try:
                k = int(nm.split("[")[-1].split("]")[0])
                if 0 <= k < K:
                    sig2[k] = float(val)
            except:
                pass
    return np.asarray(consts, dtype=float), np.asarray(sig2, dtype=float)

def get_regime_probs(res, K, mode="smoothed"):
    if mode == "smoothed":
        pm = getattr(res, "smoothed_marginal_probabilities", None)
        if pm is None:
            pm = res.filtered_marginal_probabilities
    else:
        pm = res.filtered_marginal_probabilities

    if hasattr(pm, "iloc"):
        p_last = np.asarray(pm.iloc[-1].values, dtype=float)
    else:
        pm = np.asarray(pm, dtype=float)
        if pm.ndim != 2:
            return None
        if pm.shape[1] == K:
            p_last = pm[-1, :]
        elif pm.shape[0] == K:
            p_last = pm[:, -1]
        else:
            return None

    p_last = np.where(np.isfinite(p_last), p_last, 0.0)
    s = float(np.sum(p_last))
    if s <= 0:
        p_last = np.ones(K, dtype=float) / K
    else:
        p_last = p_last / s
    return p_last

def reconstruct_transition_matrix(res, K, eps=1e-12):
    names = list(res.model.param_names)
    params = np.asarray(res.params, dtype=float)
    pat = re.compile(r"p\[(\d+)->(\d+)\]")

    row_dict = {i: {} for i in range(K)}
    for nm, val in zip(names, params):
        m = pat.search(nm.replace(" ", ""))
        if m:
            i = int(m.group(1)); j = int(m.group(2))
            if 0 <= i < K and 0 <= j < K:
                row_dict[i][j] = float(val)

    P = np.full((K, K), np.nan, dtype=float)
    for i in range(K):
        for j, v in row_dict[i].items():
            P[i, j] = v

        missing_js = [j for j in range(K) if not np.isfinite(P[i, j])]
        if len(missing_js) == 1:
            jmiss = missing_js[0]
            known_sum = np.nansum(P[i, :])
            P[i, jmiss] = 1.0 - known_sum

        if not np.isfinite(P[i, :]).all():
            P[i, :] = 1.0 / K

        P[i, :] = np.clip(P[i, :], eps, 1.0)
        P[i, :] = P[i, :] / np.sum(P[i, :])

    return P

def hqic_from(llf, nobs, k_params):
    if nobs <= 1 or not np.isfinite(llf) or not np.isfinite(k_params):
        return np.nan
    return float(-2.0*llf + 2.0*k_params*np.log(np.log(nobs)))

def summarize_series_stats(x, label_prefix):
    x = pd.Series(x).dropna()
    if x.empty:
        return {f"{label_prefix}_n": 0}
    return {
        f"{label_prefix}_n": int(x.shape[0]),
        f"{label_prefix}_min": float(x.min()),
        f"{label_prefix}_p01": float(x.quantile(0.01)),
        f"{label_prefix}_median": float(x.median()),
        f"{label_prefix}_p99": float(x.quantile(0.99)),
        f"{label_prefix}_max": float(x.max()),
        f"{label_prefix}_mean": float(x.mean()),
        f"{label_prefix}_std": float(x.std(ddof=0)),
    }

# ============================================================
# DLR baseline (rolling OLS) on same window
# ============================================================
def rolling_dlr(dfw):
    """
    OLS: dE = a + b*dG
    returns a,b,r2 or None
    """
    y = np.asarray(dfw["dE"].values, dtype=float)
    x = np.asarray(dfw["dG"].values, dtype=float)
    good = np.isfinite(y) & np.isfinite(x)
    y = y[good]; x = x[good]
    if len(y) < 50:
        return None

    # conditioning guard consistent with MSDR guard
    if np.nanstd(x) < DG_STD_MIN:
        return None
    if float(np.mean(np.abs(x) > DG_EPS)) < DG_MIN_FRAC_BIG:
        return None

    X = np.column_stack([np.ones_like(x), x])
    try:
        beta, *_ = np.linalg.lstsq(X, y, rcond=None)
        yhat = X @ beta
        ss_res = float(np.sum((y - yhat)**2))
        ss_tot = float(np.sum((y - np.mean(y))**2))
        r2 = 1.0 - ss_res/ss_tot if ss_tot > 0 else np.nan
        a = float(beta[0]); b = float(beta[1])
        return a, b, r2
    except Exception:
        return None

# ============================================================
# 1) FIND INPUT FILES
# ============================================================
gen_2023_tail_path = _find_one([
    os.path.join(INPUT_DIR, "genmix_actual_last month of 2023_tier2.3*.csv"),
    os.path.join(INPUT_DIR, "*last*month*2023*tier2.3*.csv"),
])

gen_actual_2024_path = _find_one([
    os.path.join(INPUT_DIR, "genmix_actual_2024_tier2.3*.csv"),
    os.path.join(INPUT_DIR, "*genmix*actual*2024*tier2.3*.csv"),
])

gen_forecast_2024_path = _find_one([
    os.path.join(INPUT_DIR, "genmix_tier1forecast_2024_tier2.3*.csv"),
    os.path.join(INPUT_DIR, "*genmix*tier1*forecast*2024*tier2.3*.csv"),
])

ef_2023_path = str(EMBER_SOURCE / "gv2-tier2-ember-per-tech-emission-2023.csv")
ef_2024_path = str(EMBER_SOURCE / "gv2-tier2-ember-per-tech-emission-2024.csv")

aef_2024_path = str(TIER2_WORK / "tier2_1_aef/tier2_1_aef_eval_tier1_vs_true_2024_series.csv")

print("[MEF2.3_v6_modified] Using files:")
print("  gen_2023_tail:", gen_2023_tail_path)
print("  gen_actual_2024:", gen_actual_2024_path)
print("  gen_forecast_2024:", gen_forecast_2024_path)
print("  ef_2023:", ef_2023_path)
print("  ef_2024:", ef_2024_path)
print("  aef_2024:", aef_2024_path)

# ============================================================
# 2) LOAD EF TABLES
# ============================================================
ef_2023 = load_ef(ef_2023_path, "2023")
ef_2024 = load_ef(ef_2024_path, "2024")
efmap_2023 = dict(zip(ef_2023["tech"], ef_2023["ef_kg_per_mwh"]))
efmap_2024 = dict(zip(ef_2024["tech"], ef_2024["ef_kg_per_mwh"]))

ef_2023.to_csv(os.path.join(OUT_CSV_DIR, "ember_ef_2023_loaded.csv"), index=False)
ef_2024.to_csv(os.path.join(OUT_CSV_DIR, "ember_ef_2024_loaded.csv"), index=False)

EF_MAX = float(pd.Series(list(efmap_2024.values())).max())
pd.DataFrame([{"EF_MAX_kg_per_mwh_2024": EF_MAX}]).to_csv(
    os.path.join(OUT_CSV_DIR, "ef_max_2024_for_capping.csv"), index=False
)

# ============================================================
# 3) LOAD AEF SERIES (kg/kWh)
# ============================================================
df_aef = to_utc_index_robust(pd.read_csv(aef_2024_path), col="datetime_utc")
need_aef_cols = ["aef_true_kg_per_kwh", "aef_tier1_forecast_kg_per_kwh"]
missing = [c for c in need_aef_cols if c not in df_aef.columns]
if missing:
    raise ValueError(f"AEF file missing columns: {missing}. Found: {list(df_aef.columns)}")

df_aef = df_aef[need_aef_cols].copy()
df_aef.reset_index().rename(columns={"index":"datetime_utc"}).to_csv(
    os.path.join(OUT_CSV_DIR, "aef_2024_loaded.csv"), index=False
)

# ============================================================
# 4) LOAD GENMIX, FIX TIME, CHECK MISSING HOURS, CLIP NEGATIVES
# ============================================================
df_2023_tail = to_utc_index_robust(pd.read_csv(gen_2023_tail_path), col="datetime_utc")
df_act_2024  = to_utc_index_robust(pd.read_csv(gen_actual_2024_path), col="datetime_utc")
df_for_2024  = to_utc_index_robust(pd.read_csv(gen_forecast_2024_path), col="datetime_utc")

gen_cols_2023 = [c for c in df_2023_tail.columns if c.startswith("gen_")]
gen_cols_act  = [c for c in df_act_2024.columns  if c.startswith("gen_")]
gen_cols_for  = [c for c in df_for_2024.columns  if c.startswith("gen_")]

if not gen_cols_2023 or not gen_cols_act or not gen_cols_for:
    raise ValueError("Could not find gen_* columns in one of the genmix files.")

pd.DataFrame({"gen_cols_2023_tail": gen_cols_2023}).to_csv(os.path.join(OUT_CSV_DIR, "gen_cols_2023_tail.csv"), index=False)
pd.DataFrame({"gen_cols_actual_2024": gen_cols_act}).to_csv(os.path.join(OUT_CSV_DIR, "gen_cols_actual_2024.csv"), index=False)
pd.DataFrame({"gen_cols_forecast_2024": gen_cols_for}).to_csv(os.path.join(OUT_CSV_DIR, "gen_cols_forecast_2024.csv"), index=False)

df_2023_tail, _ = complete_hourly_index(df_2023_tail, "genmix_2023_tail")
df_act_2024,  _ = complete_hourly_index(df_act_2024,  "genmix_actual_2024")
df_for_2024,  _ = complete_hourly_index(df_for_2024,  "genmix_forecast_2024")

df_2023_tail = clip_negative_generation(df_2023_tail, gen_cols_2023, "actual_2023_tail")
df_act_2024  = clip_negative_generation(df_act_2024,  gen_cols_act,  "actual_2024")
df_for_2024  = clip_negative_generation(df_for_2024,  gen_cols_for,  "forecast_2024")

# save cleaned 2024 genmix for fossil-share analysis
df_act_2024.reset_index().rename(columns={"index":"datetime_utc"}).to_csv(os.path.join(OUT_CSV_DIR, "genmix_actual_2024_clean.csv"), index=False)
df_for_2024.reset_index().rename(columns={"index":"datetime_utc"}).to_csv(os.path.join(OUT_CSV_DIR, "genmix_forecast_2024_clean.csv"), index=False)

# fossil share series (2024 only)
fossil_share_actual, fossil_cols_actual = fossil_share_from_genmix(df_act_2024, gen_cols_act, "actual_2024")
fossil_share_forec,  fossil_cols_forec  = fossil_share_from_genmix(df_for_2024, gen_cols_for, "forecast_2024")

# ============================================================
# 5) BUILD CONTIGUOUS SERIES FOR ROLLING WINDOWS (seed 2023 tail)
# ============================================================
seed_start = df_2023_tail.index.max() - pd.Timedelta(hours=ROLL_HOURS-1)
df_seed = df_2023_tail.loc[df_2023_tail.index >= seed_start].copy()

df_stream_actual = pd.concat([df_seed, df_act_2024], axis=0).sort_index()
df_stream_forec  = pd.concat([df_seed, df_for_2024], axis=0).sort_index()

# ============================================================
# 6) COMPUTE (E_t, G_t, ΔE_t, ΔG_t)
# ============================================================
def build_E_G_deltas(df_stream, gen_cols, label):
    G = df_stream[gen_cols].sum(axis=1)

    E_list = []
    miss_rows = []
    for ts, row in df_stream.iterrows():
        ef_map = efmap_2023 if int(ts.year) == 2023 else efmap_2024
        e_val, nmiss = compute_emissions_one_row(row, gen_cols, ef_map)
        E_list.append(e_val)
        miss_rows.append({"datetime_utc": str(ts), "year": int(ts.year), "n_missing_ef": int(nmiss)})

    E = pd.Series(E_list, index=df_stream.index, name="E_kgco2_per_h")
    G = pd.Series(G.values, index=df_stream.index, name="G_mw")

    dE = E.diff()
    dG = G.diff()

    out = pd.DataFrame({"E_kgco2_per_h": E, "G_mw": G, "dE": dE, "dG": dG}).dropna()

    pd.DataFrame(miss_rows).to_csv(os.path.join(OUT_CSV_DIR, f"ef_coverage_over_time_{label}.csv"), index=False)
    out.reset_index().rename(columns={"index":"datetime_utc"}).to_csv(os.path.join(OUT_CSV_DIR, f"E_G_deltas_{label}.csv"), index=False)

    return out

series_actual = build_E_G_deltas(df_stream_actual, gen_cols_act, "actual_stream")
series_forec  = build_E_G_deltas(df_stream_forec,  gen_cols_for, "forecast_stream")

# ============================================================
# 7) TARGET HOURS (selectable loop)
# ============================================================
full_hours_2024 = pd.date_range(
    pd.Timestamp(f"{APPLY_YEAR}-01-01 00:00:00", tz="UTC"),
    pd.Timestamp(f"{APPLY_YEAR}-12-31 23:00:00", tz="UTC"),
    freq="H",
    tz="UTC"
)

def make_target_hours():
    if TARGET_HOURS_MODE == "full_year":
        return full_hours_2024
    if TARGET_HOURS_MODE == "first_n":
        return full_hours_2024[:int(TARGET_FIRST_N)]
    if TARGET_HOURS_MODE == "random_n":
        rng = np.random.default_rng(TARGET_RANDOM_SEED)
        idx = rng.choice(len(full_hours_2024), size=int(TARGET_RANDOM_N), replace=False)
        return full_hours_2024[np.sort(idx)]
    if TARGET_HOURS_MODE == "explicit_list":
        if not TARGET_EXPLICIT_LIST:
            raise ValueError("TARGET_EXPLICIT_LIST is empty.")
        return pd.to_datetime(TARGET_EXPLICIT_LIST, utc=True)
    if TARGET_HOURS_MODE == "range_slice":
        a = pd.to_datetime(TARGET_RANGE_START, utc=True)
        b = pd.to_datetime(TARGET_RANGE_END, utc=True)
        return full_hours_2024[(full_hours_2024 >= a) & (full_hours_2024 <= b)]
    raise ValueError(f"Unknown TARGET_HOURS_MODE: {TARGET_HOURS_MODE}")

target_hours = make_target_hours()
pd.DataFrame({"datetime_utc": target_hours.astype(str)}).to_csv(
    os.path.join(OUT_CSV_DIR, "target_hours_used.csv"), index=False
)
print(f"[MEF2.3_v6_modified] Target hours: {len(target_hours)} (mode={TARGET_HOURS_MODE})")

# ============================================================
# 8) ROLLING MSDR + DLR baseline + exports
# ============================================================
def maybe_dump_summary(res, label, K, t_str, dump_counter, is_success=True):
    if SUMMARY_DUMP_MODE == "none":
        return dump_counter
    if SUMMARY_DUMP_MODE == "all":
        do = True
    else:
        # "some": dump every SUMMARY_DUMP_EVERY-th successful window, up to MAX
        do = is_success and (dump_counter < SUMMARY_DUMP_MAX) and ((dump_counter == 0) or (dump_counter % SUMMARY_DUMP_EVERY == 0))
    if not do:
        return dump_counter

    fname = f"summary_{label}_K{K}_{t_str.replace(':','-').replace(' ','_')}.txt"
    fpath = os.path.join(OUT_TEXT_DIR, fname)
    try:
        with open(fpath, "w") as f:
            f.write(res.summary().as_text())
        dump_counter += 1
    except Exception:
        pass
    return dump_counter

def rolling_msdr_and_dlr(series_df, label, K):
    rows_mef = []
    rows_fit = []
    rows_reg = []
    rows_tm  = []
    rows_dlr = []

    dump_counter = 0

    for idx, t in enumerate(target_hours):
        w_end   = t - pd.Timedelta(hours=1)
        w_start = t - pd.Timedelta(hours=ROLL_HOURS)
        t_str = str(t)

        if idx % 50 == 0:
            print(f"  [{label},K={K}] processed {idx}/{len(target_hours)} windows...")

        dfw = series_df.loc[(series_df.index >= w_start) & (series_df.index <= w_end)].copy()

        # ---- DLR baseline (rolling OLS) ----
        dlr_out = rolling_dlr(dfw) if dfw.shape[0] >= 50 else None
        if dlr_out is None:
            rows_dlr.append({
                "datetime_utc": t_str,
                "status": "dlr_failed_or_ill_conditioned",
                "n_obs_window": int(dfw.shape[0]),
                "dlr_intercept_kgco2_per_h": np.nan,
                "dlr_slope_kg_per_mwh": np.nan,
                "dlr_r2": np.nan,
            })
        else:
            a, b, r2 = dlr_out
            rows_dlr.append({
                "datetime_utc": t_str,
                "status": "ok",
                "n_obs_window": int(dfw.shape[0]),
                "dlr_intercept_kgco2_per_h": a,
                "dlr_slope_kg_per_mwh": b,
                "dlr_r2": r2,
            })

        # ---- MSDR ----
        if dfw.shape[0] < 50:
            rows_fit.append({"datetime_utc": t_str, "K": K, "status": "insufficient_window_rows", "n_obs_window": int(dfw.shape[0])})
            continue

        res, status = safe_fit_msdr(dfw["dE"].values, dfw["dG"].values, K=K)
        if res is None:
            rows_fit.append({"datetime_utc": t_str, "K": K, "status": status, "n_obs_window": int(dfw.shape[0])})
            continue

        dump_counter = maybe_dump_summary(res, label, K, t_str, dump_counter, is_success=True)

        # slopes / params
        slopes = extract_slopes(res, K)
        consts, sig2 = extract_intercepts_sigmas(res, K)

        # smoothed probs (paper-style interpretation)
        p_last = get_regime_probs(res, K, mode=PROB_MODE)
        if p_last is None or not np.isfinite(p_last).all():
            p_last = np.ones(K, dtype=float) / K

        k_hat = int(np.argmax(p_last))
        mef_hard = float(slopes[k_hat])
        mef_soft = float(np.dot(p_last, slopes))

        # transition matrix full reconstruction
        P = reconstruct_transition_matrix(res, K)

        # fit diagnostics
        llf = float(res.llf)
        aic = float(res.aic)
        bic = float(res.bic)
        nobs = int(getattr(res, "nobs", dfw.shape[0]))
        k_params = int(len(res.params))
        hqic = float(getattr(res, "hqic", np.nan))
        if not np.isfinite(hqic):
            hqic = hqic_from(llf, nobs, k_params)

        rows_fit.append({
            "datetime_utc": t_str, "K": K, "status": "ok",
            "n_obs_window": nobs, "log_likelihood": llf, "aic": aic, "bic": bic, "hqic": hqic, "params_count": k_params,
            "prob_mode": PROB_MODE
        })

        # main MEF + probabilities
        row_mef = {
            "datetime_utc": t_str,
            "K": K,
            "mef_soft_kg_per_mwh_raw": mef_soft,
            "mef_hard_kg_per_mwh_raw": mef_hard,
            "hard_regime": k_hat,
            **{f"p_{PROB_MODE}_regime_{k}": float(p_last[k]) for k in range(K)},
            **{f"slope_regime_{k}_kg_per_mwh": float(slopes[k]) for k in range(K)},
        }
        rows_mef.append(row_mef)

        # regime parameter rows (Table-3 style)
        for k in range(K):
            rows_reg.append({
                "datetime_utc": t_str, "K": K, "regime": k,
                "const": float(consts[k]),
                "slope_kg_per_mwh": float(slopes[k]),
                "sigma2": float(sig2[k]),
            })

        # transition matrix row (flattened)
        row_tm = {"datetime_utc": t_str, "K": K}
        for i in range(K):
            for j in range(K):
                row_tm[f"P_{i}{j}"] = float(P[i, j])
        rows_tm.append(row_tm)

    df_mef = pd.DataFrame(rows_mef)
    df_fit = pd.DataFrame(rows_fit)
    df_reg = pd.DataFrame(rows_reg)
    df_tm  = pd.DataFrame(rows_tm)
    df_dlr = pd.DataFrame(rows_dlr)

    df_mef.to_csv(os.path.join(OUT_CSV_DIR, f"mef_{label}_K{K}_raw.csv"), index=False)
    df_fit.to_csv(os.path.join(OUT_CSV_DIR, f"fit_diagnostics_{label}_K{K}.csv"), index=False)
    df_reg.to_csv(os.path.join(OUT_CSV_DIR, f"regime_params_{label}_K{K}.csv"), index=False)
    df_tm.to_csv(os.path.join(OUT_CSV_DIR, f"transition_matrix_{label}_K{K}_FULL.csv"), index=False)
    df_dlr.to_csv(os.path.join(OUT_CSV_DIR, f"dlr_baseline_{label}_window{ROLL_HOURS}h.csv"), index=False)

    return df_mef, df_fit, df_reg, df_tm, df_dlr

results = {}
for stream_label, series_df in [("actual", series_actual), ("forecast", series_forec)]:
    for K in K_LIST:
        print(f"[MEF2.3_v6_modified] Rolling MSDR+DLR: stream={stream_label}, K={K}, prob_mode={PROB_MODE}")
        results[(stream_label, K)] = rolling_msdr_and_dlr(series_df, label=stream_label, K=K)

# ============================================================
# 9) POST-PROCESS: CAP + UNIT CONVERT + AEF FILL (NaN/negative)
# ============================================================
def attach_aef_and_postprocess(df_mef_raw, stream_label, K):
    if df_mef_raw.empty:
        return df_mef_raw.copy()

    d = df_mef_raw.copy()
    d["datetime_utc"] = pd.to_datetime(d["datetime_utc"], utc=True)
    d = d.sort_values("datetime_utc").set_index("datetime_utc")

    d = d.join(df_aef, how="left")
    aef_col = "aef_true_kg_per_kwh" if stream_label == "actual" else "aef_tier1_forecast_kg_per_kwh"
    d["aef_used_kg_per_kwh"] = d[aef_col].astype(float)

    soft_raw = d["mef_soft_kg_per_mwh_raw"].astype(float)
    hard_raw = d["mef_hard_kg_per_mwh_raw"].astype(float)

    # quantiles
    q_soft_lo  = float(soft_raw.quantile(CAP_Q_LOW))
    q_soft_hi  = float(soft_raw.quantile(CAP_Q_HIGH))
    q_hard_lo  = float(hard_raw.quantile(CAP_Q_LOW))
    q_hard_hi  = float(hard_raw.quantile(CAP_Q_HIGH))

    # absolute bounds
    abs_lo = -CAP_ABS_FACTOR * EF_MAX
    abs_hi =  CAP_ABS_FACTOR * EF_MAX

    # fixed bounding: intersection (tighten)
    soft_cap_lo = max(q_soft_lo, abs_lo)
    soft_cap_hi = min(q_soft_hi, abs_hi)
    hard_cap_lo = max(q_hard_lo, abs_lo)
    hard_cap_hi = min(q_hard_hi, abs_hi)

    if soft_cap_lo > soft_cap_hi:
        soft_cap_lo, soft_cap_hi = abs_lo, abs_hi
    if hard_cap_lo > hard_cap_hi:
        hard_cap_lo, hard_cap_hi = abs_lo, abs_hi

    d["mef_soft_kg_per_mwh_capped"] = soft_raw.clip(lower=soft_cap_lo, upper=soft_cap_hi)
    d["mef_hard_kg_per_mwh_capped"] = hard_raw.clip(lower=hard_cap_lo, upper=hard_cap_hi)

    d["soft_was_capped"] = (d["mef_soft_kg_per_mwh_capped"] != d["mef_soft_kg_per_mwh_raw"])
    d["hard_was_capped"] = (d["mef_hard_kg_per_mwh_capped"] != d["mef_hard_kg_per_mwh_raw"])

    # convert to kg/kWh
    d["mef_soft_kg_per_kwh_raw"]    = d["mef_soft_kg_per_mwh_raw"] / 1000.0
    d["mef_hard_kg_per_kwh_raw"]    = d["mef_hard_kg_per_mwh_raw"] / 1000.0
    d["mef_soft_kg_per_kwh_capped"] = d["mef_soft_kg_per_mwh_capped"] / 1000.0
    d["mef_hard_kg_per_kwh_capped"] = d["mef_hard_kg_per_mwh_capped"] / 1000.0

    soft_c = d["mef_soft_kg_per_kwh_capped"].astype(float).values
    hard_c = d["mef_hard_kg_per_kwh_capped"].astype(float).values
    aef_u  = d["aef_used_kg_per_kwh"].astype(float).values

    d["soft_is_nan_pre_fill"] = ~np.isfinite(soft_c)
    d["hard_is_nan_pre_fill"] = ~np.isfinite(hard_c)
    d["soft_is_negative_pre_fill"] = np.isfinite(soft_c) & (soft_c < 0)
    d["hard_is_negative_pre_fill"] = np.isfinite(hard_c) & (hard_c < 0)

    # report exact timestamps
    pd.DataFrame({"datetime_utc": d.index[d["soft_is_nan_pre_fill"]]}).to_csv(
        os.path.join(OUT_CSV_DIR, f"{stream_label}_K{K}_soft_nan_hours_pre_fill.csv"), index=False
    )
    pd.DataFrame({"datetime_utc": d.index[d["hard_is_nan_pre_fill"]]}).to_csv(
        os.path.join(OUT_CSV_DIR, f"{stream_label}_K{K}_hard_nan_hours_pre_fill.csv"), index=False
    )
    pd.DataFrame({"datetime_utc": d.index[d["soft_is_negative_pre_fill"]]}).to_csv(
        os.path.join(OUT_CSV_DIR, f"{stream_label}_K{K}_soft_negative_hours_pre_fill.csv"), index=False
    )
    pd.DataFrame({"datetime_utc": d.index[d["hard_is_negative_pre_fill"]]}).to_csv(
        os.path.join(OUT_CSV_DIR, f"{stream_label}_K{K}_hard_negative_hours_pre_fill.csv"), index=False
    )

    # Fill rule: if NaN OR negative -> AEF
    def fill(x, aef):
        x = np.asarray(x, dtype=float)
        y = x.copy()
        bad = (~np.isfinite(y)) | (y < 0)
        y[bad] = aef[bad]
        return y, bad

    soft_filled, soft_bad = fill(soft_c, aef_u)
    hard_filled, hard_bad = fill(hard_c, aef_u)

    d["mef_soft_kg_per_kwh_filled"] = soft_filled
    d["mef_hard_kg_per_kwh_filled"] = hard_filled
    d["soft_was_filled"] = soft_bad
    d["hard_was_filled"] = hard_bad

    d["mef_soft_kg_per_mwh_filled"] = d["mef_soft_kg_per_kwh_filled"] * 1000.0
    d["mef_hard_kg_per_mwh_filled"] = d["mef_hard_kg_per_kwh_filled"] * 1000.0

    diag = [{
        "stream": stream_label, "K": K,
        "prob_mode": PROB_MODE,
        "CAP_Q_LOW": CAP_Q_LOW, "CAP_Q_HIGH": CAP_Q_HIGH,
        "CAP_ABS_FACTOR": CAP_ABS_FACTOR, "EF_MAX_kg_per_mwh": EF_MAX,
        "soft_cap_lo_kg_per_mwh": soft_cap_lo, "soft_cap_hi_kg_per_mwh": soft_cap_hi,
        "hard_cap_lo_kg_per_mwh": hard_cap_lo, "hard_cap_hi_kg_per_mwh": hard_cap_hi,
        "n_rows": int(d.shape[0]),
        "n_soft_capped": int(d["soft_was_capped"].sum()),
        "n_hard_capped": int(d["hard_was_capped"].sum()),
        "n_soft_nan_pre_fill": int(d["soft_is_nan_pre_fill"].sum()),
        "n_hard_nan_pre_fill": int(d["hard_is_nan_pre_fill"].sum()),
        "n_soft_negative_pre_fill": int(d["soft_is_negative_pre_fill"].sum()),
        "n_hard_negative_pre_fill": int(d["hard_is_negative_pre_fill"].sum()),
        "n_soft_filled_total": int(d["soft_was_filled"].sum()),
        "n_hard_filled_total": int(d["hard_was_filled"].sum()),
        "aef_missing_rows": int(pd.isna(d["aef_used_kg_per_kwh"]).sum()),
    }]
    pd.DataFrame(diag).to_csv(os.path.join(OUT_CSV_DIR, f"postprocess_diagnostics_{stream_label}_K{K}.csv"), index=False)

    out = d.reset_index().rename(columns={"index":"datetime_utc"})
    out.to_csv(os.path.join(OUT_CSV_DIR, f"mef_{stream_label}_K{K}_raw_capped_filled.csv"), index=False)

    compact = out[[
        "datetime_utc",
        "mef_soft_kg_per_kwh_filled",
        "mef_hard_kg_per_kwh_filled",
        "aef_used_kg_per_kwh",
        "soft_was_capped","hard_was_capped",
        "soft_is_nan_pre_fill","hard_is_nan_pre_fill",
        "soft_is_negative_pre_fill","hard_is_negative_pre_fill",
        "soft_was_filled","hard_was_filled",
        "hard_regime",
    ]].copy()
    compact.to_csv(os.path.join(OUT_CSV_DIR, f"mef_{stream_label}_K{K}_FINAL_stage2_inputs_kg_per_kwh.csv"), index=False)

    return out

post = {}
dlr_tables = {}
for (stream_label, K), (df_mef_raw, df_fit, df_reg, df_tm, df_dlr) in results.items():
    post[(stream_label, K)] = attach_aef_and_postprocess(df_mef_raw, stream_label, K)
    dlr_tables[(stream_label, K)] = df_dlr

# ============================================================
# 10) DLR post: convert slope to kg/kWh and provide overlay vs MSDR soft
# ============================================================
def dlr_postprocess(df_dlr, stream_label, K):
    d = df_dlr.copy()
    d["datetime_utc"] = pd.to_datetime(d["datetime_utc"], utc=True)
    d = d.sort_values("datetime_utc").set_index("datetime_utc")
    d["dlr_slope_kg_per_kwh"] = d["dlr_slope_kg_per_mwh"] / 1000.0
    d = d.reset_index()
    d.to_csv(os.path.join(OUT_CSV_DIR, f"dlr_baseline_{stream_label}_K{K}_kg_per_kwh.csv"), index=False)
    return d

dlr_post = {}
for (stream_label, K), df_dlr in dlr_tables.items():
    dlr_post[(stream_label, K)] = dlr_postprocess(df_dlr, stream_label, K)

# ============================================================
# 11) STATS (MSDR)
# ============================================================
stats_rows = []
for (stream_label, K), df_out in post.items():
    if df_out.empty:
        continue
    row = {"stream": stream_label, "K": K, "prob_mode": PROB_MODE}
    row.update(summarize_series_stats(df_out["mef_soft_kg_per_kwh_raw"],  "soft_raw"))
    row.update(summarize_series_stats(df_out["mef_hard_kg_per_kwh_raw"],  "hard_raw"))
    row.update(summarize_series_stats(df_out["mef_soft_kg_per_kwh_capped"],  "soft_capped"))
    row.update(summarize_series_stats(df_out["mef_hard_kg_per_kwh_capped"],  "hard_capped"))
    row.update(summarize_series_stats(df_out["mef_soft_kg_per_kwh_filled"], "soft_filled"))
    row.update(summarize_series_stats(df_out["mef_hard_kg_per_kwh_filled"], "hard_filled"))
    row["n_soft_capped"] = int(df_out["soft_was_capped"].sum())
    row["n_hard_capped"] = int(df_out["hard_was_capped"].sum())
    row["n_soft_filled_total"] = int(df_out["soft_was_filled"].sum())
    row["n_hard_filled_total"] = int(df_out["hard_was_filled"].sum())
    row["aef_missing_rows"] = int(pd.isna(df_out["aef_used_kg_per_kwh"]).sum())
    stats_rows.append(row)

pd.DataFrame(stats_rows).to_csv(os.path.join(OUT_CSV_DIR, "msdr_mef_stats_raw_capped_filled_by_stream.csv"), index=False)

# ============================================================
# 12) PLOTS
#   - MEF hourly and daily mean (soft/hard separately)
#   - regime probability plots (hourly and daily)
#   - DLR slope plot + overlay vs MSDR soft
#   - fossil share vs MEF and fossil share vs regime probabilities (daily)
# ============================================================
def _prep_ts(df_out):
    d = df_out.copy()
    d["datetime_utc"] = pd.to_datetime(d["datetime_utc"], utc=True)
    d = d.sort_values("datetime_utc").set_index("datetime_utc")
    return d

def plot_single_ts(index, y, title, fname, ylabel):
    fig, ax = plt.subplots(figsize=(12,4))
    ax.plot(index, y, linewidth=1.2)
    apply_plot_settings(ax, title, "Time (UTC)", ylabel)
    save_fig(fig, fname)

def plot_single_daily(index, y, title, fname, ylabel):
    s = pd.Series(y, index=index).resample("D").mean()
    fig, ax = plt.subplots(figsize=(12,4))
    ax.plot(s.index, s.values, linewidth=1.5)
    apply_plot_settings(ax, title, "Date (UTC)", ylabel)
    save_fig(fig, fname)

def plot_overlay(index, y1, y2, title, fname, ylabel, lab1, lab2):
    fig, ax = plt.subplots(figsize=(12,4))
    ax.plot(index, y1, linewidth=1.2, label=lab1)
    ax.plot(index, y2, linewidth=1.2, label=lab2)
    apply_plot_settings(ax, title, "Time (UTC)", ylabel)
    ax.legend(fontsize=10)
    save_fig(fig, fname)

def plot_overlay_daily(index, y1, y2, title, fname, ylabel, lab1, lab2):
    s1 = pd.Series(y1, index=index).resample("D").mean()
    s2 = pd.Series(y2, index=index).resample("D").mean()
    fig, ax = plt.subplots(figsize=(12,4))
    ax.plot(s1.index, s1.values, linewidth=1.5, label=lab1)
    ax.plot(s2.index, s2.values, linewidth=1.5, label=lab2)
    apply_plot_settings(ax, title, "Date (UTC)", ylabel)
    ax.legend(fontsize=10)
    save_fig(fig, fname)

# --- MEF & probs plots ---
for (stream_label, K), df_out in post.items():
    if df_out.empty:
        continue
    d = _prep_ts(df_out)

    # MEF filled hourly separate
    plot_single_ts(
        d.index, d["mef_soft_kg_per_kwh_filled"].values,
        f"{stream_label.upper()} K={K} — Soft MEF FILLED (hourly)",
        f"{stream_label}_K{K}_soft_mef_filled_hourly",
        "kgCO₂/kWh"
    )
    plot_single_ts(
        d.index, d["mef_hard_kg_per_kwh_filled"].values,
        f"{stream_label.upper()} K={K} — Hard MEF FILLED (hourly)",
        f"{stream_label}_K{K}_hard_mef_filled_hourly",
        "kgCO₂/kWh"
    )

    # MEF filled daily mean separate
    plot_single_daily(
        d.index, d["mef_soft_kg_per_kwh_filled"].values,
        f"{stream_label.upper()} K={K} — Soft MEF FILLED (daily mean)",
        f"{stream_label}_K{K}_soft_mef_filled_daily_mean",
        "kgCO₂/kWh"
    )
    plot_single_daily(
        d.index, d["mef_hard_kg_per_kwh_filled"].values,
        f"{stream_label.upper()} K={K} — Hard MEF FILLED (daily mean)",
        f"{stream_label}_K{K}_hard_mef_filled_daily_mean",
        "kgCO₂/kWh"
    )

    # Probability plots (hourly + daily) per regime
    prob_cols = [c for c in d.columns if c.startswith(f"p_{PROB_MODE}_regime_")]
    for pc in prob_cols:
        kname = pc.split("_")[-1]
        plot_single_ts(
            d.index, d[pc].values,
            f"{stream_label.upper()} K={K} — Regime prob (smoothed) k={kname} (hourly)",
            f"{stream_label}_K{K}_prob_k{kname}_hourly",
            "probability"
        )
        plot_single_daily(
            d.index, d[pc].values,
            f"{stream_label.upper()} K={K} — Regime prob (smoothed) k={kname} (daily mean)",
            f"{stream_label}_K{K}_prob_k{kname}_daily_mean",
            "probability"
        )

# --- DLR plots + overlays vs MSDR soft ---
for (stream_label, K), df_dlr in dlr_post.items():
    if ("actual", K) in post and stream_label == "actual":
        ms = _prep_ts(post[(stream_label, K)])
    elif ("forecast", K) in post and stream_label == "forecast":
        ms = _prep_ts(post[(stream_label, K)])
    else:
        continue

    dd = df_dlr.copy()
    dd["datetime_utc"] = pd.to_datetime(dd["datetime_utc"], utc=True)
    dd = dd.sort_values("datetime_utc").set_index("datetime_utc")

    common = ms.index.intersection(dd.index)
    ms2 = ms.loc[common]
    dd2 = dd.loc[common]

    # DLR slope hourly
    plot_single_ts(
        common, dd2["dlr_slope_kg_per_kwh"].values,
        f"{stream_label.upper()} K={K} — DLR slope (hourly)",
        f"{stream_label}_K{K}_dlr_slope_hourly",
        "kgCO₂/kWh"
    )
    # overlay hourly (DLR vs MSDR soft)
    plot_overlay(
        common,
        dd2["dlr_slope_kg_per_kwh"].values,
        ms2["mef_soft_kg_per_kwh_filled"].values,
        f"{stream_label.upper()} K={K} — Overlay: DLR slope vs MSDR Soft MEF (hourly)",
        f"{stream_label}_K{K}_overlay_dlr_vs_msdr_soft_hourly",
        "kgCO₂/kWh",
        "DLR slope",
        "MSDR soft (filled)"
    )
    # overlay daily
    plot_overlay_daily(
        common,
        dd2["dlr_slope_kg_per_kwh"].values,
        ms2["mef_soft_kg_per_kwh_filled"].values,
        f"{stream_label.upper()} K={K} — Overlay: DLR slope vs MSDR Soft MEF (daily mean)",
        f"{stream_label}_K{K}_overlay_dlr_vs_msdr_soft_daily_mean",
        "kgCO₂/kWh",
        "DLR slope",
        "MSDR soft (filled)"
    )

# --- fossil share vs MEF and vs probs (daily) ---
def fossil_share_plots(stream_label, fossil_share_series, K):
    if (stream_label, K) not in post:
        return
    ms = _prep_ts(post[(stream_label, K)])
    fs = fossil_share_series.copy()
    # align on 2024 target hours only (ms is only target hours)
    common = ms.index.intersection(fs.index)
    if len(common) == 0:
        return

    ms = ms.loc[common]
    fs = fs.loc[common]

    # daily series
    fs_d = fs.resample("D").mean()
    mef_soft_d = pd.Series(ms["mef_soft_kg_per_kwh_filled"].values, index=common).resample("D").mean()
    mef_hard_d = pd.Series(ms["mef_hard_kg_per_kwh_filled"].values, index=common).resample("D").mean()

    # fossil share vs MEF soft daily
    plot_overlay(
        fs_d.index,
        fs_d.values,
        mef_soft_d.values,
        f"{stream_label.upper()} K={K} — Fossil share vs Soft MEF (daily mean)",
        f"fossil_share_vs_mef_{stream_label}_K{K}_soft_daily",
        "value",
        "fossil share",
        "soft MEF (kg/kWh)"
    )
    # fossil share vs MEF hard daily
    plot_overlay(
        fs_d.index,
        fs_d.values,
        mef_hard_d.values,
        f"{stream_label.upper()} K={K} — Fossil share vs Hard MEF (daily mean)",
        f"fossil_share_vs_mef_{stream_label}_K{K}_hard_daily",
        "value",
        "fossil share",
        "hard MEF (kg/kWh)"
    )

    # fossil share vs regime probs daily
    prob_cols = [c for c in ms.columns if c.startswith(f"p_{PROB_MODE}_regime_")]
    corr_rows = []
    for pc in prob_cols:
        pk = pd.Series(ms[pc].values, index=common).resample("D").mean()
        plot_overlay(
            fs_d.index,
            fs_d.values,
            pk.values,
            f"{stream_label.upper()} K={K} — Fossil share vs Prob {pc} (daily mean)",
            f"fossil_share_vs_prob_{stream_label}_K{K}_{pc}_daily",
            "value",
            "fossil share",
            pc
        )
        # correlations (daily)
        tmp = pd.concat([fs_d.rename("fossil_share"), pk.rename(pc), mef_soft_d.rename("mef_soft")], axis=1).dropna()
        if not tmp.empty:
            corr_rows.append({
                "stream": stream_label, "K": K, "prob_col": pc,
                "corr_fossil_prob": float(tmp["fossil_share"].corr(tmp[pc])),
                "corr_fossil_mef_soft": float(tmp["fossil_share"].corr(tmp["mef_soft"])),
            })

    pd.DataFrame(corr_rows).to_csv(os.path.join(OUT_CSV_DIR, f"fossil_share_correlations_{stream_label}_K{K}_daily.csv"), index=False)

# run for each stream/K
for K in K_LIST:
    fossil_share_plots("actual", fossil_share_actual, K)
    fossil_share_plots("forecast", fossil_share_forec, K)

print("MSDR outputs:", OUT_DIR)


## Selected signals for model-ready input preparation

Column selection and file naming only. Notebook 03 retains the original final AEF filling of unavailable MEF hours.


In [ ]:
import pandas as pd
SIGNAL_OUTPUT = WORK / "signals"
SIGNAL_OUTPUT.mkdir(parents=True, exist_ok=True)
aef_file = TIER2_WORK / "tier2_1_aef/tier2_1_aef_forecast_from_tier1_2024_full.csv"
aef_selected = pd.read_csv(aef_file)[["datetime_utc", "aef_tier1_forecast_kg_per_kwh"]].rename(columns={"aef_tier1_forecast_kg_per_kwh": "grid_aef_predicted_kg_per_kwh"})
aef_selected.to_csv(SIGNAL_OUTPUT / "grid_aef_hourly_2024.csv", index=False)
up_file = TIER2_WORK / "tier2_2_mef_vnew2/vnew2_k3_q20/grid_mef_hourly_2024_k3_q20.csv"
up_selected = pd.read_csv(up_file)[["datetime_utc", "grid_mef_pred_kg_per_kwh"]].rename(columns={"grid_mef_pred_kg_per_kwh": "grid_mef_upramp_pred_kg_per_kwh"})
up_selected.to_csv(SIGNAL_OUTPUT / "grid_mef_upramp_hourly_2024.csv", index=False)
soft_file = TIER2_WORK / "mef2_3_v6_modified_outputs/csv/mef_forecast_K2_FINAL_stage2_inputs_kg_per_kwh.csv"
soft_selected = pd.read_csv(soft_file)[["datetime_utc", "mef_soft_kg_per_kwh_filled"]].rename(columns={"mef_soft_kg_per_kwh_filled": "grid_mef_soft_kg_per_kwh"})
soft_selected.to_csv(SIGNAL_OUTPUT / "grid_mef_soft_hourly_2024.csv", index=False)
print("Selected signal inputs:", SIGNAL_OUTPUT)
